# SC4000 Eugene - Kaggle Runtime Prediction Scaffold

This notebook is a clean development version for the Kaggle competition **Google - Fast or Slow? Predict AI Model Runtime**.

The first goal is not to build a winning GNN. The first goal is to build a complete, understandable baseline that can:

1. Read the Kaggle `.npz` files.
2. Understand the five collections separately.
3. Convert each graph/configuration pair into tabular features.
4. Train one baseline model per collection.
5. Rank configurations for test files.
6. Export `submission.csv`.


## 1. Mental Model of the Dataset

The folder structure mixes three ideas, which is why it looks confusing at first:

- `layout` / `tile`: the compiler optimization task.
- `xla` / `nlp`: the graph family. `xla` is general XLA graph data; `nlp` is NLP/BERT-style graph data.
- `default` / `random`: the source of layout configurations. This only exists for `layout`.
- `train` / `valid` / `test`: the usual ML split.

We will treat the problem as **five separate datasets**, and train one model for each:

```text
tile:xla
layout:xla:default
layout:xla:random
layout:nlp:default
layout:nlp:random
```

Each `.npz` file is one graph. Inside each graph there are many candidate configurations. The model predicts runtime for each configuration, then sorts configurations from fastest predicted runtime to slowest predicted runtime.


## 2. Install and Import Dependencies

This cell installs only common packages. It works in Colab/cloud and should also work locally if your Python environment has internet access.


In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(package_name, import_name=None):
    """Install a package only when it is missing from the current runtime."""
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {package_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

ensure_package('numpy')
ensure_package('pandas')
ensure_package('scikit-learn', 'sklearn')
ensure_package('joblib')
ensure_package('networkx')
ensure_package('matplotlib')
ensure_package('tqdm')

# Optional boosted-tree libraries. If installation/import fails, their experiments are skipped.
try:
    ensure_package('xgboost')
    XGBOOST_AVAILABLE = True
except Exception as exc:
    print('xgboost unavailable:', exc)
    XGBOOST_AVAILABLE = False

try:
    ensure_package('lightgbm')
    LIGHTGBM_AVAILABLE = True
except Exception as exc:
    print('lightgbm unavailable:', exc)
    LIGHTGBM_AVAILABLE = False

from pathlib import Path
import gc
import os
import warnings

os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'matplotlib'))

import joblib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# Optional PyTorch GNN support. Colab usually has torch preinstalled; skip GNN if unavailable.
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception as exc:
    print('torch unavailable; skipping GNN experiments:', exc)
    TORCH_AVAILABLE = False
    torch = None
    class _MissingTorchNN:
        Module = object
    nn = _MissingTorchNN()
    F = None

if XGBOOST_AVAILABLE:
    try:
        from xgboost import XGBRegressor
    except Exception as exc:
        print('Could not import XGBRegressor:', exc)
        XGBOOST_AVAILABLE = False

if LIGHTGBM_AVAILABLE:
    try:
        from lightgbm import LGBMRegressor
    except Exception as exc:
        print('Could not import LGBMRegressor:', exc)
        LIGHTGBM_AVAILABLE = False

print('XGBoost status:', 'enabled' if XGBOOST_AVAILABLE else 'skipped')
print('LightGBM status:', 'enabled' if LIGHTGBM_AVAILABLE else 'skipped')
print('PyTorch GNN status:', 'enabled' if TORCH_AVAILABLE else 'skipped')

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)


## 3. Locate the Kaggle Data

The notebook supports the current repository layout, where this notebook lives in `Eugene/` and the shared data folder lives one level above it as `data/`. It checks common locations:

- `./data` when running from the repository root
- `../data` when running from inside `Eugene/`
- `/content/data` in Colab
- the original Kaggle folder name `predict-ai-model-runtime` as a fallback

If your folder is somewhere else, update `find_data_root()` manually.


In [ ]:
def find_data_root():
    candidates = [
        Path.cwd() / 'data',
        Path.cwd().parent / 'data',
        Path('/content/data'),
        Path.cwd() / 'predict-ai-model-runtime',
        Path.cwd().parent / 'predict-ai-model-runtime',
        Path('/content/predict-ai-model-runtime'),
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / 'npz_all' / 'npz').exists():
            return candidate
    raise FileNotFoundError('Could not find npz_all/npz. Put the Kaggle data folder in data/ or update find_data_root().')


def find_sample_submission_path(data_root):
    candidates = [
        data_root / 'sample_submission_Eugene.csv',
        data_root / 'sample_submission.csv',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return data_root / 'sample_submission.csv'


DATA_ROOT = find_data_root()
NPZ_ROOT = DATA_ROOT / 'npz_all' / 'npz'
SAMPLE_SUBMISSION_PATH = find_sample_submission_path(DATA_ROOT)

COLLECTIONS = {
    'tile:xla': NPZ_ROOT / 'tile' / 'xla',
    'layout:xla:default': NPZ_ROOT / 'layout' / 'xla' / 'default',
    'layout:xla:random': NPZ_ROOT / 'layout' / 'xla' / 'random',
    'layout:nlp:default': NPZ_ROOT / 'layout' / 'nlp' / 'default',
    'layout:nlp:random': NPZ_ROOT / 'layout' / 'nlp' / 'random',
}

print('DATA_ROOT:', DATA_ROOT)
print('NPZ_ROOT:', NPZ_ROOT)
print('sample submission path:', SAMPLE_SUBMISSION_PATH)
print('sample submission exists:', SAMPLE_SUBMISSION_PATH.exists())


## 4. Inspect Files, Splits, and Array Shapes

This is the most important first step. Before training, we verify:

- how many files exist per collection/split
- what arrays exist inside the `.npz` files
- whether the number of configurations matches the number of runtimes
- whether test runtimes are dummy zeros, which should not be used as labels


In [ ]:
def split_files(collection_name, split):
    return sorted((COLLECTIONS[collection_name] / split).glob('*.npz'))



def infer_model_family(file_stem):
    """Infer a coarse graph/model family from a TpuGraphs file stem."""
    stem = str(file_stem).lower()
    known_families = [
        'resnet', 'bert', 'albert', 'inception', 'efficientnet', 'mlperf',
        'transformer', 'retinanet', 'mask_rcnn', 'mnasnet', 'alexnet',
        'openai', 'shapemask', 'magenta', 'brax', 'ncf', 'xception',
        'electra', 'talking-heads', 'trax', 'unet', 'experts',
    ]
    for family in known_families:
        if stem.startswith(family) or family in stem:
            return family
    if len(stem) >= 24 and all(ch in '0123456789abcdef' for ch in stem[:24]):
        return 'hashed_graph_id'
    return stem.split('_')[0].split('-')[0].split('.')[0]

rows = []
for collection_name in COLLECTIONS:
    for split in ['train', 'valid', 'test']:
        files = split_files(collection_name, split)
        rows.append({
            'collection': collection_name,
            'split': split,
            'num_files': len(files),
            'example_file': files[0].name if files else None,
        })

counts_df = pd.DataFrame(rows)
display(counts_df)

if SAMPLE_SUBMISSION_PATH.exists():
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    print('sample_submission shape:', sample_submission.shape)
    display(sample_submission.head())


In [ ]:
def summarize_npz_file(npz_path):
    """Print keys, shapes, dtypes, and basic numeric ranges for one npz file."""
    print('\nFile:', npz_path)
    rows = []
    with np.load(npz_path) as data:
        for key in data.files:
            arr = data[key]
            row = {'key': key, 'shape': arr.shape, 'dtype': str(arr.dtype)}
            if np.issubdtype(arr.dtype, np.number) and arr.size > 0:
                finite = arr[np.isfinite(arr)]
                if finite.size > 0:
                    row['min'] = float(np.min(finite))
                    row['max'] = float(np.max(finite))
                    row['mean'] = float(np.mean(finite))
            rows.append(row)
    display(pd.DataFrame(rows))

for collection_name in COLLECTIONS:
    for split in ['train', 'test']:
        files = split_files(collection_name, split)
        if files:
            print('\n===', collection_name, '|', split, '===')
            summarize_npz_file(files[0])
            break


## 5. Data Checks Instead of Traditional Cleaning

This dataset does not need normal CSV cleaning. The useful checks are structural:

- `config_runtime` must have one value per configuration in train/valid.
- test `config_runtime` is all zeros, so it is a placeholder and not a label.
- layout config features use `-1` padding in `node_config_feat`; we summarize it instead of treating every padded value as a normal feature.
- runtime values are positive and highly skewed, so the baseline trains on `log1p(runtime)`.


In [ ]:
def get_num_configs(data):
    if 'config_feat' in data:
        return data['config_feat'].shape[0]
    if 'node_config_feat' in data:
        return data['node_config_feat'].shape[0]
    raise KeyError('Could not find config_feat or node_config_feat')

check_rows = []
for collection_name in COLLECTIONS:
    for split in ['train', 'valid', 'test']:
        files = split_files(collection_name, split)[:5]
        for file_path in files:
            with np.load(file_path) as data:
                n_configs = get_num_configs(data)
                runtimes = data['config_runtime'] if 'config_runtime' in data else None
                check_rows.append({
                    'collection': collection_name,
                    'split': split,
                    'file': file_path.name,
                    'n_configs': n_configs,
                    'runtime_shape': None if runtimes is None else runtimes.shape,
                    'runtime_min': None if runtimes is None else int(np.min(runtimes)),
                    'runtime_max': None if runtimes is None else int(np.max(runtimes)),
                    'runtime_matches_configs': None if runtimes is None else len(runtimes) == n_configs,
                })

display(pd.DataFrame(check_rows))


## 5A. Exploratory Data Analysis Figure Exports

These figures are meant for the written report. They are saved into `figures/` so the image files can be used directly by `experiment_report.tex`.

The EDA focuses on dataset scale, graph size, runtime skew, and graph-family imbalance because these directly affect model choice, sampling, and validation reliability. The later training-table code uses this EDA result by applying family-stratified training-file selection and runtime-stratified configuration sampling.


In [ ]:
def save_report_figure(filename, dpi=180):
    FIGURE_DIR.mkdir(exist_ok=True)
    output_path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(output_path, dpi=dpi, bbox_inches='tight')
    plt.show()
    print('Saved', output_path)

def collect_eda_frames(max_files_per_split=None, max_runtime_samples_per_file=128):
    graph_rows = []
    runtime_rows = []

    for collection_name in tqdm(COLLECTIONS, desc='EDA collections', unit='collection'):
        for split in ['train', 'valid', 'test']:
            files = split_files(collection_name, split)
            selected_files = files if max_files_per_split is None else files[:max_files_per_split]
            for file_path in tqdm(selected_files, desc=f'EDA {collection_name} {split}', unit='file', leave=False):
                with np.load(file_path) as data:
                    n_configs = get_num_configs(data)
                    node_count = int(data['node_feat'].shape[0])
                    edge_count = int(data['edge_index'].shape[0])
                    graph_rows.append({
                        'collection': collection_name,
                        'split': split,
                        'file_stem': file_path.stem,
                        'model_family': infer_model_family(file_path.stem),
                        'n_configs': n_configs,
                        'node_count': node_count,
                        'edge_count': edge_count,
                        'edge_per_node': edge_count / max(node_count, 1),
                    })

                    if split in ['train', 'valid'] and 'config_runtime' in data:
                        runtimes = data['config_runtime'].astype(np.float64)
                        runtimes = runtimes[np.isfinite(runtimes) & (runtimes > 0)]
                        if len(runtimes) > max_runtime_samples_per_file:
                            local_rng = np.random.default_rng(RANDOM_SEED)
                            runtimes = local_rng.choice(runtimes, size=max_runtime_samples_per_file, replace=False)
                        for runtime in runtimes:
                            runtime_rows.append({
                                'collection': collection_name,
                                'split': split,
                                'log_runtime': np.log1p(runtime),
                            })

    return pd.DataFrame(graph_rows), pd.DataFrame(runtime_rows)


def plot_dataset_split_counts(counts):
    pivot = counts.pivot(index='collection', columns='split', values='num_files').reindex(COLLECTIONS.keys())
    ax = pivot[['train', 'valid', 'test']].plot(kind='bar', figsize=(10, 4.8), width=0.8)
    ax.set_title('Files per collection and split')
    ax.set_xlabel('Collection')
    ax.set_ylabel('Number of graph files')
    ax.tick_params(axis='x', labelrotation=25)
    ax.legend(title='Split')
    save_report_figure('dataset_split_counts.png')


def plot_graph_size_eda(graph_eda_df):
    train_graphs = graph_eda_df[graph_eda_df['split'] == 'train'].copy()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    train_graphs.boxplot(column='node_count', by='collection', ax=axes[0], rot=25, showfliers=False)
    train_graphs.boxplot(column='edge_per_node', by='collection', ax=axes[1], rot=25, showfliers=False)
    axes[0].set_title('Node count')
    axes[1].set_title('Edges per node')
    for ax in axes:
        ax.set_xlabel('Collection')
    fig.suptitle('Training graph size distribution')
    save_report_figure('graph_size_by_collection.png')


def plot_runtime_distribution(runtime_eda_df):
    train_runtime = runtime_eda_df[runtime_eda_df['split'] == 'train'].copy()
    fig, ax = plt.subplots(figsize=(10, 4.8))
    labels = list(COLLECTIONS.keys())
    values = [train_runtime.loc[train_runtime['collection'] == name, 'log_runtime'].dropna().to_numpy() for name in labels]
    ax.boxplot(values, tick_labels=labels, showfliers=False)
    ax.set_title('Training runtime distribution after log1p transform')
    ax.set_xlabel('Collection')
    ax.set_ylabel('log1p(runtime)')
    ax.tick_params(axis='x', labelrotation=25)
    save_report_figure('runtime_distribution_by_collection.png')


def plot_family_imbalance(graph_eda_df, top_n=8):
    train_graphs = graph_eda_df[graph_eda_df['split'] == 'train'].copy()
    family_counts = (
        train_graphs.groupby(['collection', 'model_family'])
        .size()
        .reset_index(name='graph_files')
        .sort_values(['collection', 'graph_files'], ascending=[True, False])
    )
    top_counts = family_counts.groupby('collection').head(top_n).copy()
    top_counts['label'] = top_counts['collection'] + '\n' + top_counts['model_family']

    fig, ax = plt.subplots(figsize=(12, 5.2))
    ax.bar(np.arange(len(top_counts)), top_counts['graph_files'])
    ax.set_xticks(np.arange(len(top_counts)))
    ax.set_xticklabels(top_counts['label'], rotation=75, ha='right')
    ax.set_title('Top training graph families by collection')
    ax.set_ylabel('Graph files')
    save_report_figure('train_family_counts.png')
    return family_counts


graph_eda_df, runtime_eda_df = collect_eda_frames()
display(graph_eda_df.head())
display(runtime_eda_df.head())
plot_dataset_split_counts(counts_df)
plot_graph_size_eda(graph_eda_df)
plot_runtime_distribution(runtime_eda_df)
family_counts_df = plot_family_imbalance(graph_eda_df)
graph_eda_df.to_csv(FIGURE_DIR / 'graph_eda_summary.csv', index=False)
runtime_eda_df.to_csv(FIGURE_DIR / 'runtime_eda_sample.csv', index=False)
family_counts_df.to_csv(FIGURE_DIR / 'family_counts.csv', index=False)


## 6. Visualize One Sample Graph

Each `.npz` file stores a computation graph:

- `node_feat`: numeric features for each operation node
- `node_opcode`: operation type ID for each node
- `edge_index`: directed edges between nodes

In the plot, the numbers written on the circles are just node IDs, such as node `1`, node `8`, or node `14`. They are not runtimes, ranks, or configuration numbers.

`node_opcode` is the actual operation-type value stored in the data. The node color is just a visual way to display `node_opcode`, so nodes with different operation types appear in different colors. The colors do not mean fast or slow.

Some graphs have thousands of nodes, so this visualization samples a smaller subgraph. It is for understanding only; the training pipeline below does not depend on it.


In [ ]:
def load_graph_arrays(npz_path):
    with np.load(npz_path) as data:
        return {
            'node_feat': data['node_feat'],
            'node_opcode': data['node_opcode'],
            'edge_index': data['edge_index'],
        }

def sample_nodes_for_visualization(edge_index, num_nodes, max_nodes=80, seed=RANDOM_SEED):
    """Choose a readable subset of nodes while keeping connected nodes where possible."""
    if num_nodes <= max_nodes:
        return np.arange(num_nodes)

    # Start around a node with relatively high degree so the sample is not just isolated points.
    degree = np.bincount(edge_index.reshape(-1), minlength=num_nodes)
    start_node = int(np.argmax(degree))

    selected = {start_node}
    frontier = {start_node}

    # Breadth-first expansion using the directed edges as undirected links for visualization.
    while frontier and len(selected) < max_nodes:
        next_frontier = set()
        for src, dst in edge_index:
            src = int(src)
            dst = int(dst)
            if src in frontier and dst not in selected:
                next_frontier.add(dst)
            if dst in frontier and src not in selected:
                next_frontier.add(src)
        selected.update(list(next_frontier)[: max_nodes - len(selected)])
        frontier = next_frontier

    # If the graph component is too small, fill the rest randomly.
    if len(selected) < max_nodes:
        local_rng = np.random.default_rng(seed)
        remaining = np.setdiff1d(np.arange(num_nodes), np.array(sorted(selected)))
        fill = local_rng.choice(remaining, size=min(max_nodes - len(selected), len(remaining)), replace=False)
        selected.update(fill.tolist())

    return np.array(sorted(selected), dtype=int)

def visualize_npz_graph(npz_path, max_nodes=80, title=None):
    arrays = load_graph_arrays(npz_path)
    node_opcode = arrays['node_opcode']
    edge_index = arrays['edge_index']
    num_nodes = len(node_opcode)

    selected_nodes = sample_nodes_for_visualization(edge_index, num_nodes, max_nodes=max_nodes)
    selected_set = set(selected_nodes.tolist())

    graph = nx.DiGraph()
    for node_id in selected_nodes:
        graph.add_node(int(node_id), opcode=int(node_opcode[node_id]))

    for src, dst in edge_index:
        src = int(src)
        dst = int(dst)
        if src in selected_set and dst in selected_set:
            graph.add_edge(src, dst)

    plt.figure(figsize=(13, 9))
    pos = nx.spring_layout(graph, seed=RANDOM_SEED, k=0.7)
    colors = [graph.nodes[n]['opcode'] for n in graph.nodes]

    nodes = nx.draw_networkx_nodes(
        graph,
        pos,
        node_color=colors,
        cmap='tab20',
        node_size=180,
        alpha=0.95,
    )
    nx.draw_networkx_edges(graph, pos, arrows=True, arrowsize=8, width=0.7, alpha=0.35)
    nx.draw_networkx_labels(graph, pos, labels={n: str(n) for n in graph.nodes}, font_size=7)
    plt.colorbar(nodes, label='node_opcode')
    plt.title(title or f'{npz_path.name}: {graph.number_of_nodes()} sampled nodes, {graph.number_of_edges()} sampled edges')
    plt.axis('off')
    plt.show()

    print('Original graph nodes:', num_nodes)
    print('Original graph edges:', len(edge_index))
    print('Displayed nodes:', graph.number_of_nodes())
    print('Displayed edges:', graph.number_of_edges())
    print('Unique opcodes in displayed subgraph:', sorted(set(colors))[:30])

# Pick a small tile graph by default because it is quick and readable.
sample_graph_path = split_files('tile:xla', 'train')[0]
visualize_npz_graph(sample_graph_path, max_nodes=80, title='Sample tile:xla training graph')


## 7. Feature Engineering

The baseline for this project is anchored to the TpuGraphs paper, not to an invented local model.

The paper reports learned cost-model baselines:

- for `layout:*`, a GraphSAGE-style graph neural network over the computation graph;
- for `tile:xla`, MLP, GCN, and GraphSAGE baselines;
- for ranking, losses such as ListMLE are more aligned with the competition objective than plain MSE.

The full GraphSAGE/GCN baselines are expensive to reproduce on laptop CPU. The CPU-runnable baseline implemented here is therefore the paper's **MLP-style baseline**: pool the graph into fixed-width numeric features, concatenate compiler-configuration features, and train an MLP to predict runtime. This is not the full GNN baseline, but it is the lightest paper-aligned baseline we can run locally.

We also keep a `simple_summary_ablation` using gradient boosting. That is **not** the paper baseline; it is only a sanity check to see whether the neural MLP baseline is behaving reasonably.

### What We Are Building Instead of a GNN

The paper's GNN baseline learns directly from the graph. Our proposed method tries to capture some of that graph information without graph-neural-network training:

```text
computation graph + compiler configuration
-> compact graph fingerprints
-> lightweight cost model
-> predicted runtime / ranking
```

This is still graph-aware because the new features are computed from `edge_index`, `node_opcode`, and local neighborhoods. It is cheaper than a GNN because the graph is converted once into fixed-width numeric vectors, then trained with normal CPU-friendly models.

### Method A: Repeated-Subgraph Features

The TpuGraphs paper suggests repeated subgraphs as a future direction. This is plausible because neural network graphs often contain repeated blocks: Transformer layers, residual blocks, repeated attention/feed-forward structures, and repeated convolutional modules.

In this notebook, we approximate repeated subgraphs using local opcode neighborhoods:

```text
node opcode
+ incoming neighbor opcodes
+ outgoing neighbor opcodes
-> local pattern id
```

If a pattern appears many times, it is likely capturing a repeated computation motif. We count these patterns into fixed hash bins. The model then receives columns such as `repeat_subgraph_bin_17`, which means "how often this local graph pattern appears in the graph".

### Method B: Weisfeiler-Lehman Fingerprints

The Weisfeiler-Lehman graph kernel paper gives a classical, non-neural way to summarize graph structure. The word "kernel" here means a graph similarity method, not a CUDA kernel or compiler kernel.

WL starts with a label for every node. Here the starting label is `node_opcode`. Then it repeatedly updates each node label using the labels of neighboring nodes:

```text
round 0: opcode
round 1: hash(opcode + neighbor opcodes)
round 2: hash(round-1 label + neighbor round-1 labels)
round 3: hash(round-2 label + neighbor round-2 labels)
```

After each round, we count how many times each hashed label appears. These counts are compact graph fingerprints. They capture local graph neighborhoods up to several hops without training a GNN.



### Data Imbalance Handling

The TpuGraphs paper notes that graph families are imbalanced: common model families can dominate training, while rarer families may be underrepresented. This matters because a model can learn to do well on frequent graph families and still generalize poorly to less common graph types.

We handle this in two lightweight ways:

1. We train separate models for the five official collections, so `tile`, `layout`, `xla`, `nlp`, `default`, and `random` data are not mixed into one model.
2. Within each collection, we infer a rough `model_family` from the file name, such as `resnet`, `bert`, `alexnet`, `inception`, or `albert`, then apply inverse-frequency sample weights during training.

The weighting is graph-family based, not row-count based. That is important because one graph can have thousands of configurations. Without weighting, common families and graphs with many sampled configs can dominate the training loss.

### Learned Cost Model Framing

The tensor-program optimization literature motivates this setup as a learned cost model: instead of measuring every compiler configuration on hardware, we predict which configurations are likely to run fastest. Our final output is still a ranking of configuration indices, so the model only needs to order configurations well; exact runtime prediction is useful mainly because sorting predicted runtimes gives a ranking.


In [ ]:
import time
import zlib


FEATURE_HASH_BINS = 128
WL_DEPTH = 3


FEATURE_EXPERIMENTS = {
    'simple_summary_ablation': {
        'use_degree_features': False,
        'use_dag_depth_features': False,
        'use_opcode_transition_features': False,
        'use_repeated_subgraph_features': False,
        'use_wl_features': False,
        'use_layout_local_graph_features': False,
    },
    'paper_mlp_baseline': {
        'use_degree_features': False,
        'use_dag_depth_features': False,
        'use_opcode_transition_features': False,
        'use_repeated_subgraph_features': False,
        'use_wl_features': False,
        'use_layout_local_graph_features': False,
    },
    'repeated_subgraph': {
        'use_degree_features': True,
        'use_dag_depth_features': True,
        'use_opcode_transition_features': True,
        'use_repeated_subgraph_features': True,
        'use_wl_features': False,
        'use_layout_local_graph_features': True,
    },
    'wl_fingerprint': {
        'use_degree_features': True,
        'use_dag_depth_features': True,
        'use_opcode_transition_features': False,
        'use_repeated_subgraph_features': False,
        'use_wl_features': True,
        'use_layout_local_graph_features': True,
    },
    'combined_compact_graph': {
        'use_degree_features': True,
        'use_dag_depth_features': True,
        'use_opcode_transition_features': True,
        'use_repeated_subgraph_features': True,
        'use_wl_features': True,
        'use_layout_local_graph_features': True,
    },
}

DEFAULT_FEATURE_PROFILE_NAME = 'combined_compact_graph'
ACTIVE_FEATURE_SETTINGS = FEATURE_EXPERIMENTS[DEFAULT_FEATURE_PROFILE_NAME]


def merge_feature_settings(feature_settings=None):
    settings = FEATURE_EXPERIMENTS['simple_summary_ablation'].copy()
    if feature_settings is None:
        settings.update(ACTIVE_FEATURE_SETTINGS)
    else:
        settings.update(feature_settings)
    return settings


def stable_hash_to_bin(value, n_bins=FEATURE_HASH_BINS):
    """Deterministic hash binning for graph patterns."""
    if not isinstance(value, bytes):
        value = str(value).encode('utf-8')
    return zlib.crc32(value) % n_bins


def safe_numeric_stats(prefix, arr):
    """Small aggregate stats. These are cheap and work for arrays of different shapes."""
    arr = np.asarray(arr)
    values = arr[np.isfinite(arr)] if np.issubdtype(arr.dtype, np.number) else np.array([])
    if values.size == 0:
        return {
            f'{prefix}_mean': 0.0,
            f'{prefix}_std': 0.0,
            f'{prefix}_min': 0.0,
            f'{prefix}_max': 0.0,
        }
    return {
        f'{prefix}_mean': float(values.mean()),
        f'{prefix}_std': float(values.std()),
        f'{prefix}_min': float(values.min()),
        f'{prefix}_max': float(values.max()),
    }


def distribution_stats(prefix, values):
    """Fixed summary columns for one-dimensional graph statistics."""
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        values = np.array([0.0])
    stats = safe_numeric_stats(prefix, values)
    for percentile in [10, 25, 50, 75, 90]:
        stats[f'{prefix}_p{percentile}'] = float(np.percentile(values, percentile))
    return stats


def build_adjacency(edge_index, node_count):
    """Return incoming and outgoing adjacency lists for a directed graph."""
    incoming = [[] for _ in range(node_count)]
    outgoing = [[] for _ in range(node_count)]
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if 0 <= src < node_count and 0 <= dst < node_count:
            outgoing[int(src)].append(int(dst))
            incoming[int(dst)].append(int(src))
    return incoming, outgoing


def degree_features(edge_index, node_count):
    """Degree summaries preserve more graph structure than edge count alone."""
    incoming, outgoing = build_adjacency(edge_index, node_count)
    in_degree = np.array([len(nodes) for nodes in incoming], dtype=np.float64)
    out_degree = np.array([len(nodes) for nodes in outgoing], dtype=np.float64)
    total_degree = in_degree + out_degree

    features = {}
    features.update(distribution_stats('in_degree', in_degree))
    features.update(distribution_stats('out_degree', out_degree))
    features.update(distribution_stats('total_degree', total_degree))
    features['source_node_frac'] = float(np.mean(in_degree == 0)) if node_count else 0.0
    features['sink_node_frac'] = float(np.mean(out_degree == 0)) if node_count else 0.0
    return features


def longest_dag_depths(edge_index, node_count, reverse=False):
    """Longest-path depth from sources. If cycles appear, unresolved nodes stay at zero."""
    if node_count == 0:
        return np.array([], dtype=np.float64)

    edges = np.asarray(edge_index, dtype=np.int64)
    if reverse:
        edges = edges[:, [1, 0]]

    incoming, outgoing = build_adjacency(edges, node_count)
    indegree = np.array([len(nodes) for nodes in incoming], dtype=np.int64)
    queue = [i for i, degree in enumerate(indegree) if degree == 0]
    depth = np.zeros(node_count, dtype=np.float64)
    head = 0

    while head < len(queue):
        node = queue[head]
        head += 1
        for nxt in outgoing[node]:
            if depth[nxt] < depth[node] + 1:
                depth[nxt] = depth[node] + 1
            indegree[nxt] -= 1
            if indegree[nxt] == 0:
                queue.append(nxt)

    return depth


def dag_depth_features(edge_index, node_count):
    """Summarize where nodes sit in the computation DAG."""
    source_depth = longest_dag_depths(edge_index, node_count, reverse=False)
    sink_depth = longest_dag_depths(edge_index, node_count, reverse=True)
    features = {}
    features.update(distribution_stats('source_depth', source_depth))
    features.update(distribution_stats('sink_depth', sink_depth))
    features['dag_longest_path_estimate'] = float(max(source_depth.max(initial=0.0), sink_depth.max(initial=0.0)))
    return features


def normalized_hash_counts(prefix, bin_ids, n_bins=FEATURE_HASH_BINS):
    counts = np.bincount(np.asarray(bin_ids, dtype=np.int64), minlength=n_bins)[:n_bins].astype(np.float64)
    total = counts.sum()
    if total > 0:
        counts /= total
    return {f'{prefix}_bin_{i}': float(value) for i, value in enumerate(counts)}


def opcode_transition_features(node_opcode, edge_index, n_bins=FEATURE_HASH_BINS):
    """Count directed opcode-to-opcode transitions along graph edges."""
    node_opcode = np.asarray(node_opcode, dtype=np.int64)
    bin_ids = []
    for src, dst in np.asarray(edge_index, dtype=np.int64):
        if 0 <= src < len(node_opcode) and 0 <= dst < len(node_opcode):
            pattern = f'{int(node_opcode[src])}>{int(node_opcode[dst])}'
            bin_ids.append(stable_hash_to_bin(pattern, n_bins))
    return normalized_hash_counts('opcode_transition', bin_ids, n_bins=n_bins)


def repeated_subgraph_features(node_opcode, edge_index, n_bins=FEATURE_HASH_BINS, max_neighbors_per_side=16):
    """Approximate repeated local subgraphs by hashing opcode neighborhoods."""
    node_opcode = np.asarray(node_opcode, dtype=np.int64)
    node_count = len(node_opcode)
    incoming, outgoing = build_adjacency(edge_index, node_count)
    bin_ids = []
    raw_patterns = []

    for node in range(node_count):
        in_ops = sorted(int(node_opcode[n]) for n in incoming[node])[:max_neighbors_per_side]
        out_ops = sorted(int(node_opcode[n]) for n in outgoing[node])[:max_neighbors_per_side]
        pattern = f'op={int(node_opcode[node])}|in={in_ops}|out={out_ops}'
        raw_patterns.append(pattern)
        bin_ids.append(stable_hash_to_bin(pattern, n_bins))

    features = normalized_hash_counts('repeat_subgraph', bin_ids, n_bins=n_bins)
    pattern_counts = pd.Series(raw_patterns).value_counts() if raw_patterns else pd.Series(dtype=np.int64)
    features['repeat_subgraph_unique_frac'] = float(len(pattern_counts) / max(node_count, 1))
    features['repeat_subgraph_max_frac'] = float(pattern_counts.iloc[0] / max(node_count, 1)) if len(pattern_counts) else 0.0
    features['repeat_subgraph_repeated_frac'] = float(np.mean(pattern_counts.to_numpy() > 1)) if len(pattern_counts) else 0.0
    return features


def wl_subtree_features(node_opcode, edge_index, depth=WL_DEPTH, n_bins=FEATURE_HASH_BINS):
    """Weisfeiler-Lehman subtree count features over opcode-labeled graph nodes."""
    node_opcode = np.asarray(node_opcode, dtype=np.int64)
    node_count = len(node_opcode)
    incoming, outgoing = build_adjacency(edge_index, node_count)
    neighbors = [sorted(set(incoming[i] + outgoing[i])) for i in range(node_count)]
    labels = [f'op_{int(op)}' for op in node_opcode]
    features = {}

    for round_id in range(depth + 1):
        bin_ids = [stable_hash_to_bin(label, n_bins) for label in labels]
        features.update(normalized_hash_counts(f'wl_round_{round_id}', bin_ids, n_bins=n_bins))
        if round_id == depth:
            break

        next_labels = []
        for node in range(node_count):
            neighbor_labels = sorted(labels[nbr] for nbr in neighbors[node])
            combined = labels[node] + '|' + '|'.join(neighbor_labels)
            next_labels.append(str(zlib.crc32(combined.encode('utf-8'))))
        labels = next_labels

    return features


def graph_level_features(data, feature_settings=None):
    """Features shared by every configuration inside the same graph file."""
    settings = merge_feature_settings(feature_settings)
    node_feat = data['node_feat']
    node_opcode = data['node_opcode']
    edge_index = data['edge_index']

    node_count = int(node_feat.shape[0])
    edge_count = int(edge_index.shape[0])

    features = {
        'node_count': node_count,
        'edge_count': edge_count,
        'edge_per_node': edge_count / max(node_count, 1),
        'opcode_unique': int(np.unique(node_opcode).size),
        'opcode_mean': float(np.mean(node_opcode)),
        'opcode_std': float(np.std(node_opcode)),
    }
    features.update(safe_numeric_stats('node_feat', node_feat))

    opcode_hist = np.bincount(node_opcode.astype(np.int64), minlength=128)[:128]
    opcode_hist = opcode_hist / max(opcode_hist.sum(), 1)
    for i, value in enumerate(opcode_hist):
        features[f'opcode_hist_{i}'] = float(value)

    if settings['use_degree_features']:
        features.update(degree_features(edge_index, node_count))
    if settings['use_dag_depth_features']:
        features.update(dag_depth_features(edge_index, node_count))
    if settings['use_opcode_transition_features']:
        features.update(opcode_transition_features(node_opcode, edge_index))
    if settings['use_repeated_subgraph_features']:
        features.update(repeated_subgraph_features(node_opcode, edge_index))
    if settings['use_wl_features']:
        features.update(wl_subtree_features(node_opcode, edge_index))

    return features


def choose_indices(n_items, max_items=None, seed=RANDOM_SEED):
    """Uniform fallback sampler for arrays without labels."""
    if max_items is None or n_items <= max_items:
        return np.arange(n_items)
    local_rng = np.random.default_rng(seed)
    return np.sort(local_rng.choice(n_items, size=max_items, replace=False))


def choose_runtime_stratified_indices(runtimes, max_items, seed=RANDOM_SEED):
    """Sample configs across runtime quantiles while always keeping fastest examples."""
    runtimes = np.asarray(runtimes, dtype=np.float64)
    valid_idx = np.flatnonzero(np.isfinite(runtimes) & (runtimes > 0))
    if max_items is None or len(valid_idx) <= max_items:
        return np.arange(len(runtimes))
    if len(valid_idx) == 0:
        return choose_indices(len(runtimes), max_items=max_items, seed=seed)

    local_rng = np.random.default_rng(seed)
    sorted_idx = valid_idx[np.argsort(runtimes[valid_idx])]
    fastest_count = max(1, int(max_items * 0.15))
    slowest_count = max(1, int(max_items * 0.05))
    selected = set(sorted_idx[:fastest_count].tolist())
    selected.update(sorted_idx[-slowest_count:].tolist())

    remaining = np.array([idx for idx in valid_idx if idx not in selected], dtype=np.int64)
    budget = max_items - len(selected)
    if budget > 0 and len(remaining) > 0:
        log_runtime = np.log1p(runtimes[remaining])
        ranked_remaining = remaining[np.argsort(log_runtime)]
        bins = np.array_split(ranked_remaining, min(10, len(ranked_remaining)))
        per_bin = max(1, budget // max(len(bins), 1))
        for bin_values in bins:
            if budget <= 0:
                break
            take = min(per_bin, len(bin_values), budget)
            chosen = local_rng.choice(bin_values, size=take, replace=False)
            selected.update(chosen.tolist())
            budget = max_items - len(selected)

    if len(selected) < max_items:
        remaining = np.array([idx for idx in valid_idx if idx not in selected], dtype=np.int64)
        if len(remaining) > 0:
            fill = local_rng.choice(remaining, size=min(max_items - len(selected), len(remaining)), replace=False)
            selected.update(fill.tolist())

    if len(selected) > max_items:
        selected = set(local_rng.choice(np.array(sorted(selected)), size=max_items, replace=False).tolist())
    return np.array(sorted(selected), dtype=np.int64)


def choose_config_indices(data, split, max_items=None, seed=RANDOM_SEED):
    """Choose configuration rows using labels when available and uniform sampling otherwise."""
    n_items = get_num_configs(data)
    if max_items is None or n_items <= max_items:
        return np.arange(n_items)
    if split in ['train', 'valid'] and 'config_runtime' in data:
        return choose_runtime_stratified_indices(data['config_runtime'], max_items=max_items, seed=seed)
    return choose_indices(n_items, max_items=max_items, seed=seed)


def masked_mean_and_std(mask, values):
    """Vectorized mean/std of node-level values over valid nodes for each config."""
    mask = np.asarray(mask, dtype=np.float64)
    values = np.asarray(values, dtype=np.float64)
    counts = np.maximum(mask.sum(axis=1), 1.0)
    mean = mask @ values / counts
    second = mask @ (values ** 2) / counts
    std = np.sqrt(np.maximum(second - mean ** 2, 0.0))
    return mean, std


def layout_config_local_graph_features(data, config_indices):
    """Graph-position summaries around layout-configurable nodes."""
    if 'node_config_feat' not in data or 'node_config_ids' not in data:
        return pd.DataFrame(index=np.arange(len(config_indices)))

    node_opcode = np.asarray(data['node_opcode'], dtype=np.float64)
    edge_index = data['edge_index']
    node_count = len(node_opcode)
    node_ids = np.asarray(data['node_config_ids'], dtype=np.int64)
    node_ids = np.clip(node_ids, 0, max(node_count - 1, 0))

    incoming, outgoing = build_adjacency(edge_index, node_count)
    in_degree = np.array([len(nodes) for nodes in incoming], dtype=np.float64)
    out_degree = np.array([len(nodes) for nodes in outgoing], dtype=np.float64)
    total_degree = in_degree + out_degree
    source_depth = longest_dag_depths(edge_index, node_count, reverse=False)
    sink_depth = longest_dag_depths(edge_index, node_count, reverse=True)

    selected = data['node_config_feat'][config_indices]
    valid_node_mask = np.any(selected != -1, axis=2)
    selected_no_pad = np.where(selected == -1, 0, selected)
    config_value_by_node = selected_no_pad.mean(axis=2)
    counts = np.maximum(valid_node_mask.sum(axis=1), 1)

    local_values = {
        'opcode': node_opcode[node_ids],
        'in_degree': in_degree[node_ids],
        'out_degree': out_degree[node_ids],
        'total_degree': total_degree[node_ids],
        'source_depth': source_depth[node_ids],
        'sink_depth': sink_depth[node_ids],
    }

    rows = {
        'layout_local_valid_node_frac': valid_node_mask.mean(axis=1),
        'layout_local_config_value_mean': config_value_by_node.sum(axis=1) / counts,
        'layout_local_config_value_std': np.sqrt(
            np.maximum(((config_value_by_node ** 2).sum(axis=1) / counts) - ((config_value_by_node.sum(axis=1) / counts) ** 2), 0.0)
        ),
    }

    mask_float = valid_node_mask.astype(np.float64)
    for name, values in local_values.items():
        mean, std = masked_mean_and_std(mask_float, values)
        rows[f'layout_local_{name}_mean'] = mean
        rows[f'layout_local_{name}_std'] = std
        rows[f'layout_local_config_x_{name}_mean'] = (config_value_by_node * values.reshape(1, -1)).sum(axis=1) / counts

    return pd.DataFrame(rows)


def config_features_from_file(data, collection_name, config_indices=None, feature_settings=None):
    """Return one DataFrame row per selected configuration."""
    settings = merge_feature_settings(feature_settings)

    if 'config_feat' in data:
        config_feat = data['config_feat']
        if config_indices is None:
            config_indices = np.arange(config_feat.shape[0])
        selected = config_feat[config_indices]
        rows = pd.DataFrame(selected, columns=[f'tile_config_feat_{i}' for i in range(selected.shape[1])])
        rows['config_feat_mean'] = selected.mean(axis=1)
        rows['config_feat_std'] = selected.std(axis=1)
        rows['config_feat_max'] = selected.max(axis=1)
        rows['config_feat_nonzero_frac'] = (selected != 0).mean(axis=1)
    else:
        node_config_feat = data['node_config_feat']
        if config_indices is None:
            config_indices = np.arange(node_config_feat.shape[0])
        selected = node_config_feat[config_indices]

        padding_frac = (selected == -1).mean(axis=(1, 2))
        selected_no_pad = np.where(selected == -1, 0, selected)

        pieces = []
        for stat_name, values in [
            ('mean', selected_no_pad.mean(axis=1)),
            ('std', selected_no_pad.std(axis=1)),
            ('min', selected_no_pad.min(axis=1)),
            ('max', selected_no_pad.max(axis=1)),
        ]:
            pieces.append(pd.DataFrame(values, columns=[f'layout_config_{stat_name}_{i}' for i in range(values.shape[1])]))
        rows = pd.concat(pieces, axis=1)
        rows['layout_config_padding_frac'] = padding_frac
        rows['num_configurable_nodes'] = node_config_feat.shape[1]

        if settings['use_layout_local_graph_features']:
            rows = pd.concat([rows, layout_config_local_graph_features(data, config_indices)], axis=1)

    rows['config_index'] = config_indices.astype(int)
    rows['is_tile_collection'] = int(collection_name.startswith('tile'))
    rows['is_layout_collection'] = int(collection_name.startswith('layout'))
    return rows


def features_for_npz_file(npz_path, collection_name, split, max_configs=None, seed=RANDOM_SEED, feature_settings=None):
    """Build a feature DataFrame for one graph file."""
    with np.load(npz_path) as data:
        n_configs = get_num_configs(data)
        config_indices = choose_config_indices(data, split, max_items=max_configs, seed=seed)

        graph_features = graph_level_features(data, feature_settings=feature_settings)
        config_df = config_features_from_file(data, collection_name, config_indices, feature_settings=feature_settings)

        for key, value in graph_features.items():
            config_df[key] = value

        config_df['collection'] = collection_name
        config_df['split'] = split
        config_df['file_stem'] = npz_path.stem

        if split in ['train', 'valid']:
            runtimes = data['config_runtime'][config_indices].astype(np.float64)
            config_df['runtime'] = runtimes

    numeric_cols = config_df.select_dtypes(include=[np.number]).columns
    config_df[numeric_cols] = config_df[numeric_cols].replace([np.inf, -np.inf], 0).fillna(0)
    return config_df


## 8. Build Training Tables

The full dataset can become very large because every graph has many configurations. `QUICK_MODE` controls training size, while `VALIDATION_PROFILE` controls validation cost.

Validation profiles:

- `quick`: fast iteration, fewer validation files and configs.
- `medium`: default before submission; more validation files with capped configs.
- `final`: broadest validation; use only when runtime is acceptable.


In [ ]:
QUICK_MODE = True
VALIDATION_PROFILE = 'medium'  # one of: 'quick', 'medium', 'final'


VALIDATION_PROFILES = {
    'quick': {
        'experiment_valid_files': 2,
        'experiment_valid_configs_per_file': 1000,
        'final_valid_files': 5,
        'final_valid_configs_per_file': 2000,
    },
    'medium': {
        'experiment_valid_files': 5,
        'experiment_valid_configs_per_file': 2500,
        'final_valid_files': 10,
        'final_valid_configs_per_file': 5000,
    },
    'final': {
        'experiment_valid_files': None,
        'experiment_valid_configs_per_file': 5000,
        'final_valid_files': None,
        'final_valid_configs_per_file': 10000,
    },
}

if VALIDATION_PROFILE not in VALIDATION_PROFILES:
    raise ValueError(f'Unknown VALIDATION_PROFILE: {VALIDATION_PROFILE}')

validation_profile = VALIDATION_PROFILES[VALIDATION_PROFILE]

if QUICK_MODE:
    MAX_TRAIN_FILES = {
        'tile:xla': 300,
        'layout:xla:default': 40,
        'layout:xla:random': 40,
        'layout:nlp:default': 60,
        'layout:nlp:random': 60,
    }
    MAX_TRAIN_CONFIGS_PER_FILE = {
        'tile:xla': 96,
        'layout:xla:default': 256,
        'layout:xla:random': 256,
        'layout:nlp:default': 256,
        'layout:nlp:random': 256,
    }
else:
    MAX_TRAIN_FILES = {name: None for name in COLLECTIONS}
    MAX_TRAIN_CONFIGS_PER_FILE = {name: None for name in COLLECTIONS}

MAX_VALID_FILES = validation_profile['final_valid_files']
MAX_VALID_CONFIGS_PER_FILE = validation_profile['final_valid_configs_per_file']

print('VALIDATION_PROFILE:', VALIDATION_PROFILE)
print('MAX_VALID_FILES:', MAX_VALID_FILES)
print('MAX_VALID_CONFIGS_PER_FILE:', MAX_VALID_CONFIGS_PER_FILE)


def select_files_for_split(collection_name, split, max_files=None, seed=RANDOM_SEED):
    """Select graph files, using graph-family stratification for training caps."""
    files = split_files(collection_name, split)
    if max_files is None or len(files) <= max_files:
        return files
    if split != 'train':
        return files[:max_files]

    grouped = {}
    for file_path in files:
        grouped.setdefault(infer_model_family(file_path.stem), []).append(file_path)

    local_rng = np.random.default_rng(seed)
    for family_files in grouped.values():
        local_rng.shuffle(family_files)

    selected = []
    families = sorted(grouped, key=lambda family: len(grouped[family]))
    while len(selected) < max_files and families:
        progressed = False
        for family in families:
            if grouped[family] and len(selected) < max_files:
                selected.append(grouped[family].pop(0))
                progressed = True
        if not progressed:
            break
    return sorted(selected)


def build_split_table(collection_name, split, max_files=None, max_configs_per_file=None, feature_settings=None):
    files = select_files_for_split(collection_name, split, max_files=max_files, seed=RANDOM_SEED)

    frames = []
    progress = tqdm(files, desc=f'{collection_name} {split} feature extraction', unit='file')
    for i, file_path in enumerate(progress):
        progress.set_postfix(file=file_path.stem[:18])
        frame = features_for_npz_file(
            file_path,
            collection_name=collection_name,
            split=split,
            max_configs=max_configs_per_file,
            seed=RANDOM_SEED + i,
            feature_settings=feature_settings,
        )
        frames.append(frame)

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def validate_feature_frame(frame, split):
    numeric_cols = frame.select_dtypes(include=[np.number]).columns
    numeric_values = frame[numeric_cols].to_numpy(dtype=np.float64, copy=False)
    assert np.isfinite(numeric_values).all(), 'Feature frame contains NaN or infinite values.'
    assert 'config_index' in frame.columns, 'config_index must be preserved for submission ranking.'
    if split in ['train', 'valid']:
        assert 'runtime' in frame.columns, 'Train/valid frames must include runtime labels.'
    return {
        'rows': len(frame),
        'columns': frame.shape[1],
        'numeric_columns': len(numeric_cols),
        'has_runtime': 'runtime' in frame.columns,
    }


# Smoke-test feature extraction on one file from each collection.
sanity_rows = []
for collection_name in COLLECTIONS:
    train_files = split_files(collection_name, 'train')
    if not train_files:
        continue
    frame = features_for_npz_file(
        train_files[0],
        collection_name=collection_name,
        split='train',
        max_configs=3,
        feature_settings=ACTIVE_FEATURE_SETTINGS,
    )
    result = validate_feature_frame(frame, split='train')
    result['collection'] = collection_name
    sanity_rows.append(result)

sanity_df = pd.DataFrame(sanity_rows)
display(sanity_df)


# Example: build one small table first to see what features look like.
example_collection = 'tile:xla'
example_train = build_split_table(
    example_collection,
    'train',
    max_files=2,
    max_configs_per_file=5,
    feature_settings=ACTIVE_FEATURE_SETTINGS,
)
print(example_train.shape)
display(example_train.head())


## 9. Validation Metrics

The competition uses different ranking metrics for different collections.

- For `tile:xla`, only the top 5 predicted configurations matter.
- For `layout:*`, the full ranking matters. We use a sampled Kendall-style score for quick validation.


In [ ]:
def tile_top5_score(y_true, y_pred):
    """Approximate the tile competition metric for one graph."""
    order = np.argsort(y_pred)
    best_true_runtime = np.min(y_true)
    best_runtime_in_predicted_top5 = np.min(y_true[order[:5]])
    return 2.0 - (best_runtime_in_predicted_top5 / best_true_runtime)


def sampled_kendall_score(y_true, y_pred, max_pairs=20000, seed=RANDOM_SEED):
    """Fast sampled Kendall-style score in [-1, 1]. Higher is better."""
    n = len(y_true)
    if n < 2:
        return np.nan
    local_rng = np.random.default_rng(seed)
    i = local_rng.integers(0, n, size=max_pairs)
    j = local_rng.integers(0, n, size=max_pairs)
    mask = i != j
    i, j = i[mask], j[mask]

    true_order = np.sign(y_true[i] - y_true[j])
    pred_order = np.sign(y_pred[i] - y_pred[j])
    useful = (true_order != 0) & (pred_order != 0)
    if useful.sum() == 0:
        return np.nan
    return float(np.mean(true_order[useful] == pred_order[useful]) * 2 - 1)


## 10. Train One Model per Collection

This section is organized around the paper-baseline story:

1. **Published reference baseline:** the TpuGraphs paper's full GNN baselines, especially GraphSAGE/GCN. We cite these, but do not fully reproduce them on laptop CPU.
2. **Implemented paper-aligned baseline:** `paper_mlp_baseline`, a CPU-runnable MLP over pooled graph/config features. This is the baseline row to compare against in this notebook.
3. **Ablation only:** `simple_summary_ablation`, a gradient-boosting model over the same simple features. This is not the project baseline.
4. **Proposed methods:** repeated-subgraph features, WL fingerprints, and the combined compact-graph method.

The final submission no longer forces one method for every collection. It chooses the best validation `ranking_score` per collection, then retrains that chosen method on the larger quick-mode training cap.

The training target is graph-centered log runtime:

```text
log1p(config_runtime) - median log1p(runtime for configs in the same graph)
```

This helps because the competition cares about ranking configurations **within the same graph**. The absolute runtime scale of different graphs is less important than whether the model can identify the faster configs for each graph.


The training loop also applies family-balanced sample weights. Rows from common graph families receive lower weight, and rows from rare graph families receive higher weight. This is a direct response to the imbalance concern discussed in the TpuGraphs paper.


In [ ]:
NON_FEATURE_COLUMNS = {'collection', 'split', 'file_stem', 'runtime', 'model_family'}


RUN_EXPERIMENT_COMPARISON = True
TARGET_MODE = 'graph_centered_log_runtime'
USE_FAMILY_BALANCING = True
USE_RANK_ENSEMBLE = True
ENSEMBLE_TOP_K = 2
ENSEMBLE_MAX_RANKING_SCORE_GAP = 0.03
RUN_GNN_EXPERIMENTS = True
GNN_COLLECTIONS = ['layout:xla:default', 'layout:xla:random']
GNN_MAX_EPOCHS = 8
GNN_BATCH_SIZE = 128
GNN_HIDDEN_DIM = 64
GNN_LEARNING_RATE = 1e-3

BASE_EXPERIMENT_NAMES = [
    'paper_mlp_baseline',
    'simple_summary_ablation',
    'repeated_subgraph_hgb',
    'wl_fingerprint_hgb',
    'combined_compact_graph_hgb',
]

OPTIONAL_EXPERIMENT_NAMES = []
if XGBOOST_AVAILABLE:
    OPTIONAL_EXPERIMENT_NAMES.extend([
        'simple_summary_xgb',
        'combined_compact_graph_xgb',
    ])
if LIGHTGBM_AVAILABLE:
    OPTIONAL_EXPERIMENT_NAMES.extend([
        'simple_summary_lgbm',
        'combined_compact_graph_lgbm',
    ])
if RUN_GNN_EXPERIMENTS and TORCH_AVAILABLE:
    OPTIONAL_EXPERIMENT_NAMES.append('layout_graphsage_gnn')

EXPERIMENT_NAMES = BASE_EXPERIMENT_NAMES + OPTIONAL_EXPERIMENT_NAMES

EXPERIMENT_MODEL_TYPES = {
    'paper_mlp_baseline': 'mlp',
    'simple_summary_ablation': 'hgb',
    'repeated_subgraph_hgb': 'hgb',
    'wl_fingerprint_hgb': 'hgb',
    'combined_compact_graph_hgb': 'hgb',
    'simple_summary_xgb': 'xgb',
    'combined_compact_graph_xgb': 'xgb',
    'simple_summary_lgbm': 'lgbm',
    'combined_compact_graph_lgbm': 'lgbm',
    'layout_graphsage_gnn': 'gnn',
}

EXPERIMENT_FEATURE_SETTINGS = {
    'paper_mlp_baseline': FEATURE_EXPERIMENTS['paper_mlp_baseline'],
    'simple_summary_ablation': FEATURE_EXPERIMENTS['simple_summary_ablation'],
    'repeated_subgraph_hgb': FEATURE_EXPERIMENTS['repeated_subgraph'],
    'wl_fingerprint_hgb': FEATURE_EXPERIMENTS['wl_fingerprint'],
    'combined_compact_graph_hgb': FEATURE_EXPERIMENTS['combined_compact_graph'],
    'simple_summary_xgb': FEATURE_EXPERIMENTS['simple_summary_ablation'],
    'combined_compact_graph_xgb': FEATURE_EXPERIMENTS['combined_compact_graph'],
    'simple_summary_lgbm': FEATURE_EXPERIMENTS['simple_summary_ablation'],
    'combined_compact_graph_lgbm': FEATURE_EXPERIMENTS['combined_compact_graph'],
    'layout_graphsage_gnn': FEATURE_EXPERIMENTS['simple_summary_ablation'],
}

print('XGBOOST_AVAILABLE:', XGBOOST_AVAILABLE)
print('LIGHTGBM_AVAILABLE:', LIGHTGBM_AVAILABLE)
print('Base experiments:', BASE_EXPERIMENT_NAMES)
print('Optional experiments enabled:', OPTIONAL_EXPERIMENT_NAMES if OPTIONAL_EXPERIMENT_NAMES else 'none')
if not XGBOOST_AVAILABLE:
    print('Skipping XGBoost experiments: xgboost is unavailable in this runtime')
if not LIGHTGBM_AVAILABLE:
    print('Skipping LightGBM experiments: lightgbm is unavailable in this runtime')
print('Final experiment run order:', EXPERIMENT_NAMES)
print('USE_RANK_ENSEMBLE:', USE_RANK_ENSEMBLE)
print('ENSEMBLE_TOP_K:', ENSEMBLE_TOP_K)
print('ENSEMBLE_MAX_RANKING_SCORE_GAP:', ENSEMBLE_MAX_RANKING_SCORE_GAP)
print('RUN_GNN_EXPERIMENTS:', RUN_GNN_EXPERIMENTS)
print('GNN_COLLECTIONS:', GNN_COLLECTIONS)
if RUN_GNN_EXPERIMENTS and not TORCH_AVAILABLE:
    print('Skipping GNN experiments: torch is unavailable in this runtime')

# Used when RUN_EXPERIMENT_COMPARISON is False, or as a fallback if a validation run fails.
# These defaults come from the first Colab experiment output and can be overwritten by the
# automatic validation-based selector below.
DEFAULT_FINAL_EXPERIMENT_BY_COLLECTION = {
    'tile:xla': 'paper_mlp_baseline',
    'layout:xla:default': 'wl_fingerprint_hgb',
    'layout:xla:random': 'wl_fingerprint_hgb',
    'layout:nlp:default': 'simple_summary_ablation',
    'layout:nlp:random': 'simple_summary_ablation',
}

EXPERIMENT_MAX_TRAIN_FILES = {
    'tile:xla': 80,
    'layout:xla:default': 12,
    'layout:xla:random': 12,
    'layout:nlp:default': 12,
    'layout:nlp:random': 12,
}
EXPERIMENT_MAX_TRAIN_CONFIGS_PER_FILE = {
    'tile:xla': 64,
    'layout:xla:default': 128,
    'layout:xla:random': 128,
    'layout:nlp:default': 128,
    'layout:nlp:random': 128,
}
EXPERIMENT_MAX_VALID_FILES = validation_profile['experiment_valid_files']
EXPERIMENT_MAX_VALID_CONFIGS_PER_FILE = validation_profile['experiment_valid_configs_per_file']
print('EXPERIMENT_MAX_VALID_FILES:', EXPERIMENT_MAX_VALID_FILES)
print('EXPERIMENT_MAX_VALID_CONFIGS_PER_FILE:', EXPERIMENT_MAX_VALID_CONFIGS_PER_FILE)

def add_model_family_column(frame):
    frame = frame.copy()
    frame['model_family'] = frame['file_stem'].map(infer_model_family)
    return frame


def family_balance_weights(train_df):
    """Inverse-frequency row weights by graph family, normalized to mean 1."""
    if not USE_FAMILY_BALANCING:
        return None
    family_counts = train_df['model_family'].value_counts()
    weights = train_df['model_family'].map(lambda family: 1.0 / family_counts[family]).to_numpy(dtype=np.float64).copy()
    weights *= len(weights) / max(weights.sum(), 1e-12)
    return weights


def fit_model_with_optional_weights(model, X_train, y_train, sample_weight):
    if sample_weight is None:
        model.fit(X_train, y_train)
        return model

    try:
        model.fit(X_train, y_train, sample_weight=sample_weight)
    except (TypeError, ValueError):
        # Pipeline-based MLP baselines may not accept sample_weight directly.
        # Try routing it to the MLP step; if sklearn still rejects it, fall back
        # to unweighted fitting so the paper-style baseline remains runnable.
        try:
            model.fit(X_train, y_train, mlpregressor__sample_weight=sample_weight)
        except (TypeError, ValueError):
            model.fit(X_train, y_train)
    return model


def make_runtime_model(model_type):
    if model_type == 'mlp':
        return make_pipeline(
            StandardScaler(),
            MLPRegressor(
                hidden_layer_sizes=(128, 64),
                activation='relu',
                solver='adam',
                alpha=1e-4,
                batch_size=256,
                learning_rate_init=1e-3,
                max_iter=120,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=10,
                random_state=RANDOM_SEED,
            ),
        )

    if model_type == 'hgb':
        return HistGradientBoostingRegressor(
            loss='squared_error',
            learning_rate=0.06,
            max_iter=250,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=RANDOM_SEED,
        )

    if model_type == 'xgb':
        if not XGBOOST_AVAILABLE:
            raise ImportError('xgboost is not available in this runtime')
        return XGBRegressor(
            objective='reg:squarederror',
            n_estimators=450,
            learning_rate=0.045,
            max_depth=6,
            min_child_weight=5,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            reg_alpha=0.0,
            tree_method='hist',
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=0,
        )

    if model_type == 'lgbm':
        if not LIGHTGBM_AVAILABLE:
            raise ImportError('lightgbm is not available in this runtime')
        return LGBMRegressor(
            objective='regression',
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            max_depth=-1,
            min_child_samples=20,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=2.0,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbose=-1,
        )

    raise ValueError(f'Unknown model_type: {model_type}')


def make_training_target(train_df, target_mode=TARGET_MODE):
    log_runtime = np.log1p(train_df['runtime'].to_numpy(dtype=np.float64))
    if target_mode == 'log_runtime':
        return log_runtime
    if target_mode == 'graph_centered_log_runtime':
        graph_median = train_df.assign(_log_runtime=log_runtime).groupby('file_stem')['_log_runtime'].transform('median')
        return log_runtime - graph_median.to_numpy(dtype=np.float64)
    raise ValueError(f'Unknown target_mode: {target_mode}')


def validation_targets_and_scores(valid_df, raw_pred, target_mode=TARGET_MODE):
    y_true_runtime = valid_df['runtime'].to_numpy(dtype=np.float64)
    if target_mode == 'log_runtime':
        pred_for_ranking = np.expm1(raw_pred)
        true_for_mae = np.log1p(y_true_runtime)
        pred_for_mae = raw_pred
    elif target_mode == 'graph_centered_log_runtime':
        pred_for_ranking = raw_pred
        true_log = np.log1p(y_true_runtime)
        true_center = valid_df.assign(_log_runtime=true_log).groupby('file_stem')['_log_runtime'].transform('median')
        true_for_mae = true_log - true_center.to_numpy(dtype=np.float64)
        pred_for_mae = raw_pred
    else:
        raise ValueError(f'Unknown target_mode: {target_mode}')
    return y_true_runtime, pred_for_ranking, true_for_mae, pred_for_mae


def evaluate_valid_files(
    model,
    feature_columns,
    collection_name,
    max_files=MAX_VALID_FILES,
    max_configs_per_file=MAX_VALID_CONFIGS_PER_FILE,
    feature_settings=None,
    target_mode=TARGET_MODE,
):
    files = split_files(collection_name, 'valid')
    if max_files is not None:
        files = files[:max_files]

    scores = []
    maes = []
    progress = tqdm(files, desc=f'{collection_name} validation', unit='file')
    for i, file_path in enumerate(progress):
        progress.set_postfix(file=file_path.stem[:18])
        valid_df = features_for_npz_file(
            file_path,
            collection_name=collection_name,
            split='valid',
            max_configs=max_configs_per_file,
            seed=RANDOM_SEED + i,
            feature_settings=feature_settings,
        )
        X_valid = valid_df.reindex(columns=feature_columns, fill_value=0)
        raw_pred = model.predict(X_valid)
        y_true, y_pred_for_ranking, y_true_for_mae, y_pred_for_mae = validation_targets_and_scores(
            valid_df,
            raw_pred,
            target_mode=target_mode,
        )

        maes.append(mean_absolute_error(y_true_for_mae, y_pred_for_mae))
        if collection_name.startswith('tile'):
            scores.append(tile_top5_score(y_true, y_pred_for_ranking))
        else:
            scores.append(sampled_kendall_score(y_true, y_pred_for_ranking, seed=RANDOM_SEED + i))

    return {
        'collection': collection_name,
        'valid_files': len(files),
        'target_mode': target_mode,
        'log_runtime_mae': float(np.mean(maes)) if maes else np.nan,
        'ranking_score': float(np.nanmean(scores)) if scores else np.nan,
    }




def graph_centered_targets_for_runtimes(runtimes):
    log_runtime = np.log1p(np.asarray(runtimes, dtype=np.float64))
    return log_runtime - np.median(log_runtime)


class SimpleGraphSAGEModel(nn.Module):
    """Small GraphSAGE-style network for layout config ranking."""

    def __init__(self, node_dim, config_dim, hidden_dim=GNN_HIDDEN_DIM, opcode_vocab_size=256, opcode_emb_dim=16):
        super().__init__()
        self.opcode_embedding = nn.Embedding(opcode_vocab_size, opcode_emb_dim)
        input_dim = node_dim + config_dim + opcode_emb_dim
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.sage1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.sage2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def sage_step(self, h, adj, layer):
        neigh = torch.einsum('ij,bjh->bih', adj, h)
        updated = F.relu(layer(torch.cat([h, neigh], dim=-1)))
        return updated + h

    def forward(self, base_node, opcode, config_node, adj):
        batch_size = config_node.shape[0]
        base = base_node.unsqueeze(0).expand(batch_size, -1, -1)
        opcode_emb = self.opcode_embedding(opcode).unsqueeze(0).expand(batch_size, -1, -1)
        h = F.relu(self.input_proj(torch.cat([base, config_node, opcode_emb], dim=-1)))
        h = self.sage_step(h, adj, self.sage1)
        h = self.sage_step(h, adj, self.sage2)
        pooled = torch.cat([h.mean(dim=1), h.max(dim=1).values], dim=-1)
        return self.head(pooled).squeeze(-1)


class LayoutGraphSAGERegressor:
    """Plain-PyTorch GNN wrapper with fit/predict methods used by the notebook pipeline."""

    def __init__(self, hidden_dim=GNN_HIDDEN_DIM, lr=GNN_LEARNING_RATE, epochs=GNN_MAX_EPOCHS, batch_size=GNN_BATCH_SIZE, seed=RANDOM_SEED):
        self.hidden_dim = hidden_dim
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.seed = seed
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = None
        self.node_dim = None
        self.config_dim = None
        self.feature_count = 0
        self.train_rows = 0

    def _normalized_adjacency(self, edge_index, node_count):
        adj = np.eye(node_count, dtype=np.float32)
        for src, dst in np.asarray(edge_index, dtype=np.int64):
            if 0 <= src < node_count and 0 <= dst < node_count:
                adj[int(src), int(dst)] = 1.0
                adj[int(dst), int(src)] = 1.0
        denom = np.maximum(adj.sum(axis=1, keepdims=True), 1.0)
        return adj / denom

    def _base_node_features(self, data):
        node_feat = np.asarray(data['node_feat'], dtype=np.float32)
        node_feat = np.log1p(np.maximum(node_feat, 0.0))
        mean = node_feat.mean(axis=0, keepdims=True)
        std = node_feat.std(axis=0, keepdims=True) + 1e-6
        return (node_feat - mean) / std

    def _prepare_graph_tensors(self, data):
        node_count = int(data['node_feat'].shape[0])
        base_node = self._base_node_features(data)
        opcode = np.asarray(data['node_opcode'], dtype=np.int64)
        opcode = np.clip(opcode, 0, 255)
        adj = self._normalized_adjacency(data['edge_index'], node_count)
        return (
            torch.tensor(base_node, dtype=torch.float32, device=self.device),
            torch.tensor(opcode, dtype=torch.long, device=self.device),
            torch.tensor(adj, dtype=torch.float32, device=self.device),
        )

    def _config_node_tensor(self, data, config_indices):
        node_count = int(data['node_feat'].shape[0])
        node_ids = np.asarray(data['node_config_ids'], dtype=np.int64)
        node_ids = np.clip(node_ids, 0, max(node_count - 1, 0))
        selected = np.asarray(data['node_config_feat'][config_indices], dtype=np.float32)
        selected = np.where(selected == -1, 0.0, selected)
        config_node = np.zeros((len(config_indices), node_count, selected.shape[2]), dtype=np.float32)
        config_node[:, node_ids, :] = selected
        return torch.tensor(config_node, dtype=torch.float32, device=self.device)

    def _ensure_model(self, data):
        node_dim = int(data['node_feat'].shape[1])
        config_dim = int(data['node_config_feat'].shape[2])
        if self.model is None:
            torch.manual_seed(self.seed)
            self.node_dim = node_dim
            self.config_dim = config_dim
            self.feature_count = node_dim + config_dim + 16
            self.model = SimpleGraphSAGEModel(node_dim, config_dim, hidden_dim=self.hidden_dim).to(self.device)
        elif node_dim != self.node_dim or config_dim != self.config_dim:
            raise ValueError(f'GNN dimension mismatch: expected node_dim={self.node_dim}, config_dim={self.config_dim}; got node_dim={node_dim}, config_dim={config_dim}')

    def fit_files(self, collection_name, files, max_configs_per_file=None, target_mode=TARGET_MODE):
        if not TORCH_AVAILABLE:
            raise ImportError('torch is unavailable')
        if not files:
            raise ValueError(f'No training files for {collection_name}')
        print('GNN device:', self.device)

        first_file = files[0]
        with np.load(first_file) as data:
            self._ensure_model(data)
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-4)
        self.model.train()

        total_rows = 0
        for epoch in range(1, self.epochs + 1):
            epoch_losses = []
            progress = tqdm(files, desc=f'{collection_name} GNN epoch {epoch}/{self.epochs}', unit='file', leave=False)
            for file_id, file_path in enumerate(progress):
                with np.load(file_path) as data:
                    config_indices = choose_config_indices(data, 'train', max_items=max_configs_per_file, seed=self.seed + file_id)
                    runtimes = np.asarray(data['config_runtime'], dtype=np.float64)[config_indices]
                    target = graph_centered_targets_for_runtimes(runtimes)
                    base_node, opcode, adj = self._prepare_graph_tensors(data)
                    order = np.arange(len(config_indices))
                    np.random.default_rng(self.seed + epoch + file_id).shuffle(order)

                    for start_idx in range(0, len(order), self.batch_size):
                        batch_pos = order[start_idx:start_idx + self.batch_size]
                        batch_config_indices = config_indices[batch_pos]
                        y = torch.tensor(target[batch_pos], dtype=torch.float32, device=self.device)
                        config_node = self._config_node_tensor(data, batch_config_indices)
                        pred = self.model(base_node, opcode, config_node, adj)
                        loss = F.smooth_l1_loss(pred, y)
                        optimizer.zero_grad()
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                        optimizer.step()
                        epoch_losses.append(float(loss.detach().cpu()))
                    total_rows += len(config_indices)
                    progress.set_postfix(loss=np.mean(epoch_losses[-10:]) if epoch_losses else np.nan)
            print(f'{collection_name} GNN epoch {epoch}/{self.epochs} mean_loss={np.mean(epoch_losses):.5f}')
        self.train_rows = int(total_rows / max(self.epochs, 1))
        return self

    def predict_file(self, file_path, collection_name=None, max_configs=None, seed=RANDOM_SEED, split='test'):
        self.model.eval()
        with np.load(file_path) as data:
            config_indices = choose_config_indices(data, split, max_items=max_configs, seed=seed)
            base_node, opcode, adj = self._prepare_graph_tensors(data)
            preds = []
            with torch.no_grad():
                for start_idx in range(0, len(config_indices), self.batch_size):
                    batch_indices = config_indices[start_idx:start_idx + self.batch_size]
                    config_node = self._config_node_tensor(data, batch_indices)
                    pred = self.model(base_node, opcode, config_node, adj).detach().cpu().numpy()
                    preds.append(pred)
        return config_indices, np.concatenate(preds) if preds else np.array([], dtype=np.float64)


def evaluate_gnn_valid_files(model, collection_name, max_files=MAX_VALID_FILES, max_configs_per_file=MAX_VALID_CONFIGS_PER_FILE, target_mode=TARGET_MODE):
    files = split_files(collection_name, 'valid')
    if max_files is not None:
        files = files[:max_files]
    scores = []
    maes = []
    progress = tqdm(files, desc=f'{collection_name} GNN validation', unit='file')
    for i, file_path in enumerate(progress):
        with np.load(file_path) as data:
            config_indices = choose_config_indices(data, 'valid', max_items=max_configs_per_file, seed=RANDOM_SEED + i)
            y_true = np.asarray(data['config_runtime'], dtype=np.float64)[config_indices]
            _, raw_pred = model.predict_file(file_path, collection_name=collection_name, max_configs=max_configs_per_file, seed=RANDOM_SEED + i, split='valid')
            true_for_mae = graph_centered_targets_for_runtimes(y_true)
            maes.append(mean_absolute_error(true_for_mae, raw_pred))
            scores.append(sampled_kendall_score(y_true, raw_pred, seed=RANDOM_SEED + i))
    return {
        'collection': collection_name,
        'valid_files': len(files),
        'target_mode': target_mode,
        'log_runtime_mae': float(np.mean(maes)) if maes else np.nan,
        'ranking_score': float(np.nanmean(scores)) if scores else np.nan,
    }


def train_layout_gnn_collection(collection_name, max_train_files, max_train_configs_per_file, max_valid_files, max_valid_configs_per_file, target_mode=TARGET_MODE):
    start = time.perf_counter()
    if collection_name not in GNN_COLLECTIONS:
        raise ValueError(f'GNN experiment is restricted to {GNN_COLLECTIONS}, got {collection_name}')
    files = select_files_for_split(collection_name, 'train', max_files=max_train_files, seed=RANDOM_SEED)
    family_counts = pd.Series([infer_model_family(file_path.stem) for file_path in files]).value_counts().to_dict()
    print('Graph families:', family_counts)

    model = LayoutGraphSAGERegressor()
    print(f'Fitting gnn model for {collection_name}: files={len(files)}, max_configs_per_file={max_train_configs_per_file}')
    fit_start = time.perf_counter()
    model.fit_files(collection_name, files, max_configs_per_file=max_train_configs_per_file, target_mode=target_mode)
    print(f'Finished fitting gnn model for {collection_name} in {time.perf_counter() - fit_start:.2f}s')

    validation = evaluate_gnn_valid_files(
        model,
        collection_name,
        max_files=max_valid_files,
        max_configs_per_file=max_valid_configs_per_file,
        target_mode=target_mode,
    )
    validation['model_type'] = 'gnn'
    validation['family_balancing'] = False
    validation['family_count'] = len(family_counts)
    validation['feature_count'] = model.feature_count
    validation['train_rows'] = model.train_rows
    validation['train_seconds'] = round(time.perf_counter() - start, 2)
    gc.collect()
    return model, ['__layout_graphsage_gnn__'], validation


def train_collection_model(
    collection_name,
    feature_settings,
    model_type,
    max_train_files,
    max_train_configs_per_file,
    max_valid_files,
    max_valid_configs_per_file,
    target_mode=TARGET_MODE,
):
    if model_type == 'gnn':
        return train_layout_gnn_collection(
            collection_name,
            max_train_files=max_train_files,
            max_train_configs_per_file=max_train_configs_per_file,
            max_valid_files=max_valid_files,
            max_valid_configs_per_file=max_valid_configs_per_file,
            target_mode=target_mode,
        )

    start = time.perf_counter()
    train_df = build_split_table(
        collection_name,
        'train',
        max_files=max_train_files,
        max_configs_per_file=max_train_configs_per_file,
        feature_settings=feature_settings,
    )

    train_df = add_model_family_column(train_df)
    family_counts = train_df[['file_stem', 'model_family']].drop_duplicates()['model_family'].value_counts().to_dict()
    print('Graph families:', family_counts)

    feature_columns = [col for col in train_df.columns if col not in NON_FEATURE_COLUMNS]
    X_train = train_df[feature_columns]
    y_train = make_training_target(train_df, target_mode=target_mode)
    sample_weight = family_balance_weights(train_df)

    model = make_runtime_model(model_type)
    fit_start = time.perf_counter()
    print(f'Fitting {model_type} model for {collection_name}: rows={len(X_train)}, features={len(feature_columns)}')
    model = fit_model_with_optional_weights(model, X_train, y_train, sample_weight)
    print(f'Finished fitting {model_type} model for {collection_name} in {time.perf_counter() - fit_start:.2f}s')

    validation = evaluate_valid_files(
        model,
        feature_columns,
        collection_name,
        max_files=max_valid_files,
        max_configs_per_file=max_valid_configs_per_file,
        feature_settings=feature_settings,
        target_mode=target_mode,
    )
    validation['model_type'] = model_type
    validation['family_balancing'] = USE_FAMILY_BALANCING
    validation['family_count'] = train_df['model_family'].nunique()
    validation['feature_count'] = len(feature_columns)
    validation['train_rows'] = len(train_df)
    validation['train_seconds'] = round(time.perf_counter() - start, 2)

    del train_df, X_train, y_train
    gc.collect()
    return model, feature_columns, validation


def choose_final_experiments(experiment_results_df):
    if experiment_results_df.empty:
        return DEFAULT_FINAL_EXPERIMENT_BY_COLLECTION.copy()

    selected = {}
    for collection_name in COLLECTIONS:
        candidates = experiment_results_df[experiment_results_df['collection'] == collection_name].copy()
        if candidates.empty:
            selected[collection_name] = DEFAULT_FINAL_EXPERIMENT_BY_COLLECTION[collection_name]
            continue

        # Ranking is the competition objective. Feature count breaks near-ties toward simpler models.
        candidates = candidates.sort_values(
            ['ranking_score', 'feature_count'],
            ascending=[False, True],
        )
        selected[collection_name] = candidates.iloc[0]['experiment']
    return selected


def choose_final_ensemble_experiments(experiment_results_df, top_k=ENSEMBLE_TOP_K, max_score_gap=ENSEMBLE_MAX_RANKING_SCORE_GAP):
    """Choose top validated experiments per collection for rank averaging."""
    if not USE_RANK_ENSEMBLE or top_k <= 1 or experiment_results_df.empty:
        return {
            collection_name: [experiment_name]
            for collection_name, experiment_name in choose_final_experiments(experiment_results_df).items()
        }

    selected = {}
    fallback = DEFAULT_FINAL_EXPERIMENT_BY_COLLECTION.copy()
    for collection_name in COLLECTIONS:
        candidates = experiment_results_df[experiment_results_df['collection'] == collection_name].copy()
        candidates = candidates[np.isfinite(candidates['ranking_score'])]
        if candidates.empty:
            selected[collection_name] = [fallback[collection_name]]
            continue

        candidates = candidates.sort_values(
            ['ranking_score', 'feature_count'],
            ascending=[False, True],
        )
        best_score = float(candidates.iloc[0]['ranking_score'])
        if max_score_gap is not None:
            candidates = candidates[candidates['ranking_score'] >= best_score - max_score_gap]

        experiments = candidates['experiment'].drop_duplicates().head(top_k).tolist()
        selected[collection_name] = experiments or [fallback[collection_name]]
    return selected


experiment_results = []

if RUN_EXPERIMENT_COMPARISON:
    for experiment_name in tqdm(EXPERIMENT_NAMES, desc='Experiment comparison', unit='experiment'):
        print('\n' + '#' * 80)
        print('Experiment:', experiment_name)
        feature_settings = EXPERIMENT_FEATURE_SETTINGS[experiment_name]
        model_type = EXPERIMENT_MODEL_TYPES[experiment_name]
        collections_for_experiment = GNN_COLLECTIONS if model_type == 'gnn' else COLLECTIONS

        for collection_name in tqdm(collections_for_experiment, desc=f'{experiment_name} collections', unit='collection', leave=False):
            print('\n' + '=' * 80)
            print('Training collection:', collection_name)
            _, _, validation = train_collection_model(
                collection_name,
                feature_settings=feature_settings,
                model_type=model_type,
                max_train_files=EXPERIMENT_MAX_TRAIN_FILES[collection_name],
                max_train_configs_per_file=EXPERIMENT_MAX_TRAIN_CONFIGS_PER_FILE[collection_name],
                max_valid_files=EXPERIMENT_MAX_VALID_FILES,
                max_valid_configs_per_file=EXPERIMENT_MAX_VALID_CONFIGS_PER_FILE,
                target_mode=TARGET_MODE,
            )
            validation['experiment'] = experiment_name
            validation['baseline_reference'] = experiment_name == 'paper_mlp_baseline'
            experiment_results.append(validation)
            display(pd.DataFrame(experiment_results))

experiment_results_df = pd.DataFrame(experiment_results)
if not experiment_results_df.empty:
    summary_columns = [
        'experiment',
        'baseline_reference',
        'model_type',
        'collection',
        'target_mode',
        'family_balancing',
        'family_count',
        'ranking_score',
        'log_runtime_mae',
        'feature_count',
        'train_rows',
        'train_seconds',
    ]
    display(experiment_results_df[summary_columns].sort_values(['collection', 'experiment']))

    baseline_scores = (
        experiment_results_df[experiment_results_df['experiment'] == 'paper_mlp_baseline']
        [['collection', 'ranking_score', 'log_runtime_mae']]
        .rename(columns={
            'ranking_score': 'paper_mlp_ranking_score',
            'log_runtime_mae': 'paper_mlp_log_runtime_mae',
        })
    )
    comparison_to_paper_mlp = experiment_results_df.merge(baseline_scores, on='collection', how='left')
    comparison_to_paper_mlp['ranking_score_delta_vs_paper_mlp'] = (
        comparison_to_paper_mlp['ranking_score'] - comparison_to_paper_mlp['paper_mlp_ranking_score']
    )
    comparison_to_paper_mlp['log_runtime_mae_delta_vs_paper_mlp'] = (
        comparison_to_paper_mlp['log_runtime_mae'] - comparison_to_paper_mlp['paper_mlp_log_runtime_mae']
    )
    display(
        comparison_to_paper_mlp[
            [
                'experiment',
                'collection',
                'ranking_score_delta_vs_paper_mlp',
                'log_runtime_mae_delta_vs_paper_mlp',
            ]
        ].sort_values(['collection', 'experiment'])
    )


# Train the final submission models. Each collection can use one winner or a small rank ensemble.
final_experiment_by_collection = choose_final_experiments(experiment_results_df)
final_ensemble_experiments_by_collection = choose_final_ensemble_experiments(experiment_results_df)

display(
    pd.DataFrame(
        [
            {
                'collection': collection_name,
                'selected_final_experiment': final_experiment_by_collection[collection_name],
                'rank_ensemble_experiments': ', '.join(final_ensemble_experiments_by_collection[collection_name]),
                'ensemble_size': len(final_ensemble_experiments_by_collection[collection_name]),
            }
            for collection_name in COLLECTIONS
        ]
    )
)

models = {}
feature_columns_by_collection = {}
feature_settings_by_collection = {}
model_type_by_collection = {}
ensemble_members_by_collection = {}
validation_rows = []

print('\n' + '#' * 80)
print('Training final per-collection selected models')
print('Rank ensemble mode:', 'enabled' if USE_RANK_ENSEMBLE else 'disabled')

for collection_name in tqdm(COLLECTIONS, desc='Final model training', unit='collection'):
    ensemble_experiments = final_ensemble_experiments_by_collection[collection_name]
    ensemble_members_by_collection[collection_name] = []

    print('\n' + '=' * 80)
    print('Training collection:', collection_name)
    print('Selected final experiment:', final_experiment_by_collection[collection_name])
    print('Rank ensemble experiments:', ensemble_experiments)

    for member_rank, final_experiment_name in enumerate(ensemble_experiments, start=1):
        feature_settings = EXPERIMENT_FEATURE_SETTINGS[final_experiment_name]
        model_type = EXPERIMENT_MODEL_TYPES[final_experiment_name]

        print('\n' + '-' * 80)
        print(f'Training ensemble member {member_rank}/{len(ensemble_experiments)}:', final_experiment_name)

        model, feature_columns, validation = train_collection_model(
            collection_name,
            feature_settings=feature_settings,
            model_type=model_type,
            max_train_files=MAX_TRAIN_FILES[collection_name],
            max_train_configs_per_file=MAX_TRAIN_CONFIGS_PER_FILE[collection_name],
            max_valid_files=MAX_VALID_FILES,
            max_valid_configs_per_file=MAX_VALID_CONFIGS_PER_FILE,
            target_mode=TARGET_MODE,
        )

        member = {
            'experiment': final_experiment_name,
            'model_type': model_type,
            'model': model,
            'feature_columns': feature_columns,
            'feature_settings': feature_settings,
        }
        ensemble_members_by_collection[collection_name].append(member)

        # Keep first member in the old dictionaries so diagnostics and saved-model code remain compatible.
        if member_rank == 1:
            models[collection_name] = model
            feature_columns_by_collection[collection_name] = feature_columns
            feature_settings_by_collection[collection_name] = feature_settings
            model_type_by_collection[collection_name] = model_type

        validation['experiment'] = final_experiment_name
        validation['ensemble_member_rank'] = member_rank
        validation['ensemble_size'] = len(ensemble_experiments)
        validation_rows.append(validation)
        display(pd.DataFrame(validation_rows))

validation_df = pd.DataFrame(validation_rows)
display(validation_df)


## 10A. Report Model Diagnostic Figure Exports

This cell saves the model comparison, final validation, prediction diagnostic, and overfitting diagnostic figures used by the report. Run it after the experiment-comparison and final-model training cell.

Training loss alone only shows whether optimization is still improving on the training data. To reason about overfitting, compare it with validation behaviour: MLP validation score for neural models, and staged validation MAE for HGB models.


In [ ]:
def plot_metric_by_experiment(results_df, metric, filename, higher_is_better=True):
    if results_df.empty or metric not in results_df.columns:
        print(f'Skipping {filename}: missing {metric}')
        return
    pivot = results_df.pivot(index='collection', columns='experiment', values=metric).reindex(COLLECTIONS.keys())
    ordered_cols = [name for name in EXPERIMENT_NAMES if name in pivot.columns]
    ax = pivot[ordered_cols].plot(kind='bar', figsize=(13, 5.4), width=0.82)
    direction = 'higher is better' if higher_is_better else 'lower is better'
    ax.set_title(f'{metric} by experiment ({direction})')
    ax.set_xlabel('Collection')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', labelrotation=25)
    ax.legend(title='Experiment', bbox_to_anchor=(1.02, 1), loc='upper left')
    save_report_figure(filename)


def plot_final_validation_metrics(validation_results_df):
    if validation_results_df.empty:
        print('Skipping final validation figure: validation_df is empty')
        return
    plot_df = validation_results_df.copy()
    if 'ensemble_member_rank' in plot_df.columns:
        plot_df = plot_df[plot_df['ensemble_member_rank'] == 1]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    ordered = plot_df.set_index('collection').reindex(COLLECTIONS.keys()).reset_index()
    axes[0].bar(ordered['collection'], ordered['ranking_score'])
    axes[0].set_title('Final ranking score')
    axes[0].set_ylabel('Ranking score')
    axes[1].bar(ordered['collection'], ordered['log_runtime_mae'])
    axes[1].set_title('Final log-runtime MAE')
    axes[1].set_ylabel('MAE')
    for ax in axes:
        ax.tick_params(axis='x', labelrotation=25)
        ax.set_xlabel('Collection')
    save_report_figure('final_validation_metrics.png')


def get_mlp_regressor(model):
    if hasattr(model, 'named_steps') and 'mlpregressor' in model.named_steps:
        return model.named_steps['mlpregressor']
    if isinstance(model, MLPRegressor):
        return model
    return None


def plot_mlp_training_loss(models_by_collection):
    loss_curves = []
    validation_curves = []
    for collection_name, model in models_by_collection.items():
        mlp = get_mlp_regressor(model)
        if mlp is None:
            continue
        if hasattr(mlp, 'loss_curve_'):
            for iteration, loss in enumerate(mlp.loss_curve_, start=1):
                loss_curves.append({'collection': collection_name, 'iteration': iteration, 'loss': loss})
        if hasattr(mlp, 'validation_scores_'):
            for iteration, score in enumerate(mlp.validation_scores_, start=1):
                validation_curves.append({'collection': collection_name, 'iteration': iteration, 'validation_score': score})

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    if loss_curves:
        loss_df = pd.DataFrame(loss_curves)
        for collection_name, group in loss_df.groupby('collection'):
            axes[0].plot(group['iteration'], group['loss'], marker='o', markersize=3, linewidth=1.5, label=collection_name)
        axes[0].legend(title='Collection')
    else:
        axes[0].text(0.5, 0.5, 'No final MLP loss curve available', ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('MLP training loss')
    axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Training loss')

    if validation_curves:
        validation_df = pd.DataFrame(validation_curves)
        for collection_name, group in validation_df.groupby('collection'):
            axes[1].plot(group['iteration'], group['validation_score'], marker='o', markersize=3, linewidth=1.5, label=collection_name)
        axes[1].legend(title='Collection')
    else:
        axes[1].text(0.5, 0.5, 'No MLP validation score available', ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('MLP validation score')
    axes[1].set_xlabel('Iteration')
    axes[1].set_ylabel('Validation score')
    save_report_figure('mlp_training_loss.png')


def collect_hgb_validation_curves(max_files_per_collection=1, max_configs_per_file=1500, step=5):
    rows = []
    for collection_name, model in tqdm(models.items(), desc='HGB validation curves', unit='collection'):
        if not hasattr(model, 'staged_predict'):
            continue

        files = split_files(collection_name, 'valid')[:max_files_per_collection]
        if not files:
            continue
        feature_columns = feature_columns_by_collection[collection_name]
        feature_settings = feature_settings_by_collection[collection_name]

        for file_id, file_path in enumerate(tqdm(files, desc=f'{collection_name} HGB curve files', unit='file', leave=False)):
            valid_df = features_for_npz_file(
                file_path,
                collection_name=collection_name,
                split='valid',
                max_configs=max_configs_per_file,
                seed=RANDOM_SEED + file_id,
                feature_settings=feature_settings,
            )
            X_valid = valid_df.reindex(columns=feature_columns, fill_value=0)
            dummy_pred = np.zeros(len(valid_df), dtype=np.float64)
            _, _, y_true_for_mae, _ = validation_targets_and_scores(valid_df, dummy_pred, target_mode=TARGET_MODE)

            staged_predictions = model.staged_predict(X_valid)
            total_iterations = getattr(model, 'n_iter_', None)
            for iteration, staged_pred in enumerate(tqdm(staged_predictions, desc=f'{collection_name} staged predict', total=total_iterations, unit='iter', leave=False), start=1):
                if iteration == 1 or iteration % step == 0:
                    rows.append({
                        'collection': collection_name,
                        'file_stem': file_path.stem,
                        'iteration': iteration,
                        'validation_mae': mean_absolute_error(y_true_for_mae, staged_pred),
                    })
    return pd.DataFrame(rows)


def plot_hgb_validation_curves(hgb_curves_df):
    fig, ax = plt.subplots(figsize=(9, 5.0))
    if hgb_curves_df.empty:
        ax.text(0.5, 0.5, 'No HGB staged validation curve available', ha='center', va='center', transform=ax.transAxes)
    else:
        summary = hgb_curves_df.groupby(['collection', 'iteration'], as_index=False)['validation_mae'].mean()
        for collection_name, group in summary.groupby('collection'):
            ax.plot(group['iteration'], group['validation_mae'], linewidth=1.8, label=collection_name)
            best = group.loc[group['validation_mae'].idxmin()]
            ax.scatter([best['iteration']], [best['validation_mae']], s=35)
            ax.annotate(
                f"best {int(best['iteration'])}",
                xy=(best['iteration'], best['validation_mae']),
                xytext=(4, 4),
                textcoords='offset points',
                fontsize=8,
            )
        ax.legend(title='Collection')
    ax.set_title('HGB staged validation MAE')
    ax.set_xlabel('Boosting iteration')
    ax.set_ylabel('Validation MAE on centered log runtime')
    save_report_figure('hgb_validation_mae_curve.png')


def collect_prediction_diagnostics(max_files_per_collection=1, max_configs_per_file=1500):
    rows = []
    for collection_name, model in tqdm(models.items(), desc='Prediction diagnostics', unit='collection'):
        files = split_files(collection_name, 'valid')[:max_files_per_collection]
        feature_columns = feature_columns_by_collection[collection_name]
        feature_settings = feature_settings_by_collection[collection_name]
        for i, file_path in enumerate(tqdm(files, desc=f'{collection_name} diagnostic files', unit='file', leave=False)):
            if model_type_by_collection.get(collection_name) == 'gnn':
                with np.load(file_path) as data:
                    config_indices = choose_config_indices(data, 'valid', max_items=max_configs_per_file, seed=RANDOM_SEED + i)
                    y_true_runtime = np.asarray(data['config_runtime'], dtype=np.float64)[config_indices]
                _, raw_pred = model.predict_file(file_path, collection_name=collection_name, max_configs=max_configs_per_file, seed=RANDOM_SEED + i, split='valid')
                y_true_for_mae = graph_centered_targets_for_runtimes(y_true_runtime)
                y_pred_for_mae = raw_pred
                file_stem = file_path.stem
            else:
                valid_df = features_for_npz_file(
                    file_path,
                    collection_name=collection_name,
                    split='valid',
                    max_configs=max_configs_per_file,
                    seed=RANDOM_SEED + i,
                    feature_settings=feature_settings,
                )
                X_valid = valid_df.reindex(columns=feature_columns, fill_value=0)
                raw_pred = model.predict(X_valid)
                _, _, y_true_for_mae, y_pred_for_mae = validation_targets_and_scores(valid_df, raw_pred, target_mode=TARGET_MODE)
                file_stem = file_path.stem
            sample_size = min(500, len(y_true_for_mae))
            sample_idx = np.linspace(0, len(y_true_for_mae) - 1, sample_size).astype(int)
            for idx in sample_idx:
                rows.append({
                    'collection': collection_name,
                    'file_stem': file_stem,
                    'true_centered_log_runtime': float(y_true_for_mae[idx]),
                    'pred_centered_log_runtime': float(y_pred_for_mae[idx]),
                })
    return pd.DataFrame(rows)


def plot_prediction_diagnostics(prediction_diagnostics_df):
    if prediction_diagnostics_df.empty:
        print('Skipping prediction diagnostics: no rows')
        return
    labels = list(COLLECTIONS.keys())
    fig, axes = plt.subplots(2, 3, figsize=(14, 8.5))
    axes = axes.ravel()
    for ax, collection_name in zip(axes, labels):
        group = prediction_diagnostics_df[prediction_diagnostics_df['collection'] == collection_name]
        ax.scatter(group['true_centered_log_runtime'], group['pred_centered_log_runtime'], s=9, alpha=0.35)
        if not group.empty:
            lo = min(group['true_centered_log_runtime'].min(), group['pred_centered_log_runtime'].min())
            hi = max(group['true_centered_log_runtime'].max(), group['pred_centered_log_runtime'].max())
            ax.plot([lo, hi], [lo, hi], color='black', linewidth=1, linestyle='--')
        ax.set_title(collection_name)
        ax.set_xlabel('True centered log runtime')
        ax.set_ylabel('Predicted centered log runtime')
    axes[-1].axis('off')
    save_report_figure('prediction_diagnostics.png')


plot_metric_by_experiment(experiment_results_df, 'ranking_score', 'experiment_ranking_scores.png', higher_is_better=True)
plot_metric_by_experiment(experiment_results_df, 'log_runtime_mae', 'experiment_log_runtime_mae.png', higher_is_better=False)
plot_final_validation_metrics(validation_df)
plot_mlp_training_loss(models)
hgb_validation_curves_df = collect_hgb_validation_curves()
display(hgb_validation_curves_df.head())
plot_hgb_validation_curves(hgb_validation_curves_df)
prediction_diagnostics_df = collect_prediction_diagnostics()
display(prediction_diagnostics_df.head())
plot_prediction_diagnostics(prediction_diagnostics_df)
experiment_results_df.to_csv(FIGURE_DIR / 'experiment_results_summary.csv', index=False)
validation_df.to_csv(FIGURE_DIR / 'final_validation_summary.csv', index=False)
hgb_validation_curves_df.to_csv(FIGURE_DIR / 'hgb_validation_curves.csv', index=False)
prediction_diagnostics_df.to_csv(FIGURE_DIR / 'prediction_diagnostics_sample.csv', index=False)


## 11. Save Models

Saving models lets us generate submissions later without retraining in the same runtime.


In [ ]:
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(models, MODEL_DIR / 'baseline_models.joblib')
joblib.dump(feature_columns_by_collection, MODEL_DIR / 'baseline_feature_columns.joblib')
joblib.dump(feature_settings_by_collection, MODEL_DIR / 'feature_settings_by_collection.joblib')
joblib.dump(model_type_by_collection, MODEL_DIR / 'model_type_by_collection.joblib')
joblib.dump(final_experiment_by_collection, MODEL_DIR / 'final_experiment_by_collection.joblib')
joblib.dump(final_ensemble_experiments_by_collection, MODEL_DIR / 'final_ensemble_experiments_by_collection.joblib')
joblib.dump(ensemble_members_by_collection, MODEL_DIR / 'ensemble_members_by_collection.joblib')

print('Saved models to:', MODEL_DIR.resolve())


## 12. Generate `submission.csv`

For each test file:

1. Build features for every configuration.
2. Predict runtime.
3. Sort configuration indices by predicted runtime ascending.
4. Write the ranking to `TopConfigs`.

For `tile:xla`, Kaggle only uses the first 5 indices, so we output the top 5. For `layout:*`, we output the full ranking.


In [ ]:
def top_config_count(top_configs):
    return len([value for value in str(top_configs).split(';') if value != ''])


def fit_ranking_to_expected_length(ordered_indices, expected_count, n_configs, collection_name):
    ordered_indices = [int(index) for index in ordered_indices]

    if expected_count is None:
        expected_count = 5 if collection_name.startswith('tile') else n_configs

    ordered_indices = ordered_indices[:expected_count]

    # The official template has fixed row lengths: 5 entries for tile rows and
    # 1000 entries for layout rows. Some local test files expose fewer tile
    # configs or 1001 layout configs, so match the template length exactly.
    filler = 0
    existing = set(ordered_indices)
    while len(ordered_indices) < expected_count:
        if filler not in existing:
            ordered_indices.append(filler)
            existing.add(filler)
        filler += 1

    return ordered_indices


def predict_member_ranks_for_file(file_path, collection_name, member):
    if member.get('model_type') == 'gnn':
        config_indices, pred_score = member['model'].predict_file(file_path, collection_name=collection_name, max_configs=None)
        ranks = pd.Series(pred_score).rank(method='average', ascending=True).to_numpy(dtype=np.float64)
        return pd.Series(ranks, index=config_indices.astype(np.int64))

    test_df = features_for_npz_file(
        file_path,
        collection_name=collection_name,
        split='test',
        max_configs=None,
        feature_settings=member['feature_settings'],
    )
    X_test = test_df.reindex(columns=member['feature_columns'], fill_value=0)

    # Lower predicted score means faster. Convert each model's scores to ranks before
    # averaging so models with different score scales can be ensembled safely.
    pred_score = member['model'].predict(X_test)
    ranks = pd.Series(pred_score).rank(method='average', ascending=True).to_numpy(dtype=np.float64)
    return pd.Series(ranks, index=test_df['config_index'].to_numpy(dtype=np.int64))


def predict_ranking_for_file(file_path, collection_name, expected_count=None):
    members = ensemble_members_by_collection.get(collection_name)
    if not members:
        members = [{
            'experiment': final_experiment_by_collection[collection_name],
            'model_type': model_type_by_collection[collection_name],
            'model': models[collection_name],
            'feature_columns': feature_columns_by_collection[collection_name],
            'feature_settings': feature_settings_by_collection[collection_name],
        }]

    rank_columns = []
    for member in members:
        rank_columns.append(predict_member_ranks_for_file(file_path, collection_name, member))

    rank_matrix = pd.concat(rank_columns, axis=1)
    average_rank = rank_matrix.mean(axis=1).sort_values(kind='mergesort')
    ordered_indices = average_rank.index.to_numpy(dtype=np.int64)
    ordered_indices = fit_ranking_to_expected_length(
        ordered_indices,
        expected_count=expected_count,
        n_configs=len(average_rank),
        collection_name=collection_name,
    )

    return ';'.join(str(int(i)) for i in ordered_indices)


def id_to_collection_and_stem(row_id):
    parts = row_id.split(':')
    return ':'.join(parts[:-1]), parts[-1]


def create_submission(output_path='submission.csv'):
    expected_counts_by_id = {}
    if SAMPLE_SUBMISSION_PATH.exists():
        template = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        ids = template['ID'].tolist()
        expected_counts_by_id = dict(zip(template['ID'], template['TopConfigs'].map(top_config_count)))
    else:
        ids = []
        for collection_name in COLLECTIONS:
            for file_path in split_files(collection_name, 'test'):
                ids.append(f'{collection_name}:{file_path.stem}')

    rows = []
    progress = tqdm(ids, desc='Submission prediction', unit='row')
    for row_number, row_id in enumerate(progress, start=1):
        collection_name, file_stem = id_to_collection_and_stem(row_id)
        file_path = COLLECTIONS[collection_name] / 'test' / f'{file_stem}.npz'
        progress.set_postfix(collection=collection_name, ensemble_size=len(ensemble_members_by_collection.get(collection_name, [])))
        top_configs = predict_ranking_for_file(
            file_path,
            collection_name,
            expected_count=expected_counts_by_id.get(row_id),
        )
        rows.append({'ID': row_id, 'TopConfigs': top_configs})

    submission = pd.DataFrame(rows)
    submission.to_csv(output_path, index=False)
    return submission


def validate_submission_frame(submission):
    required_columns = ['ID', 'TopConfigs']
    assert list(submission.columns) == required_columns, f'submission columns must be {required_columns}'
    assert not submission['ID'].duplicated().any(), 'submission contains duplicate IDs'
    assert submission['TopConfigs'].notna().all(), 'submission contains missing TopConfigs'

    expected_counts_by_id = {}
    if SAMPLE_SUBMISSION_PATH.exists():
        template = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        assert submission['ID'].tolist() == template['ID'].tolist(), 'submission IDs/order do not match sample_submission.csv'
        expected_counts_by_id = dict(zip(template['ID'], template['TopConfigs'].map(top_config_count)))

    for row_id, top_configs in zip(submission['ID'], submission['TopConfigs']):
        collection_name, file_stem = id_to_collection_and_stem(row_id)
        assert collection_name in COLLECTIONS, f'unknown collection in ID: {row_id}'
        file_path = COLLECTIONS[collection_name] / 'test' / f'{file_stem}.npz'
        assert file_path.exists(), f'missing test npz for ID: {row_id}'

        values = [int(value) for value in str(top_configs).split(';') if value != '']
        with np.load(file_path) as data:
            n_configs = get_num_configs(data)

        expected_count = expected_counts_by_id.get(row_id)
        if expected_count is None:
            expected_count = 5 if collection_name.startswith('tile') else n_configs

        assert values, f'empty TopConfigs for ID: {row_id}'
        assert len(values) == expected_count, f'TopConfigs length mismatch for ID: {row_id}'
        assert len(values) == len(set(values)), f'duplicate config index in TopConfigs for ID: {row_id}'

        valid_values = [value for value in values if 0 <= value < n_configs]
        if collection_name.startswith('tile'):
            assert len(valid_values) == min(expected_count, n_configs), f'tile row does not include enough valid configs for ID: {row_id}'
        else:
            assert len(valid_values) == expected_count, f'layout config index out of range for ID: {row_id}'
            if expected_count == n_configs:
                assert sorted(values) == list(range(n_configs)), f'layout row is not a full permutation for ID: {row_id}'

    return {
        'rows': len(submission),
        'columns': list(submission.columns),
        'unique_ids': int(submission['ID'].nunique()),
        'sample_submission_match': bool(SAMPLE_SUBMISSION_PATH.exists()),
    }


submission = create_submission('submission.csv')
validation_report = validate_submission_frame(submission)
print('submission shape:', submission.shape)
display(submission.head())
print('submission validation:', validation_report)
print('Wrote submission.csv')


## 13. Next Improvements

This notebook now has three levels of comparison:

1. the **published reference baseline** from the TpuGraphs paper, which uses full GNN/MLP learned cost models;
2. the **implemented paper-aligned baseline**, `paper_mlp_baseline`, which is the CPU-runnable baseline row in Section 10;
3. the **proposed compact graph methods**, which add repeated-subgraph and WL-style graph fingerprints.

The final submission can use a small rank ensemble per collection. The notebook first chooses the best validation experiment, then optionally trains the top validated experiments within a small score gap and averages their predicted ranks. It also includes an optional layout-only GraphSAGE-style GNN experiment for `layout:xla:default` and `layout:xla:random`.

Useful next steps:

1. Use `VALIDATION_PROFILE = 'medium'` before submission, and `final` only when runtime is acceptable.
2. Increase `MAX_TRAIN_FILES` and `MAX_TRAIN_CONFIGS_PER_FILE` in cloud.
3. Validate whether `USE_RANK_ENSEMBLE = True` beats the single best model for each collection; disable it if a collection gets worse.
4. Compare the optional `layout_graphsage_gnn` validation score against the tree models; keep it only if it improves `layout:xla:*`.
5. Add a true ranking objective, especially for `layout:*`.
6. Report feature-ablation results against `paper_mlp_baseline`.
7. Only consider full GraphSAGE/GCN reproduction if compute is available and the project scope requires the exact official GNN baseline.
